# Review the existing repo

In [11]:

# 1. Mount Google Drive and configure project root
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import subprocess

# Existing GitHub repository cloned in Google Drive
REPO_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

# Raw and processed data directories
RAW_DIR = REPO_ROOT / "data" / "raw" / "olist"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"

# Python source directory
SRC_DIR = REPO_ROOT / "src"

# Make project modules importable
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


# 2. Verify the project setup
print("\n" + "=" * 65)
print("1. PROJECT PATH VERIFICATION")
print("=" * 65)

print("Repository exists:", REPO_ROOT.exists())
print("Raw data directory exists:", RAW_DIR.exists())
print("Processed data directory exists:", PROCESSED_DIR.exists())
print("Source directory exists:", SRC_DIR.exists())

assert REPO_ROOT.is_dir(), (
    f"Repository not found: {REPO_ROOT}"
)


# 3. Git inspection helper
def run_git(*args):
    result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args],
        capture_output=True,
        text=True
    )

    print(f"\n$ git {' '.join(args)}")
    print(result.stdout or "(no output)")

    if result.stderr:
        print("STDERR:", result.stderr)

    if result.returncode != 0:
        print("Exit code:", result.returncode)

    return result


# 4. Check Git status
print("\n" + "=" * 65)
print("2. GIT STATUS")
print("=" * 65)

run_git("status", "--short", "--branch")


# 5. Check current branch and recent commits
print("\n" + "=" * 65)
print("3. BRANCH AND RECENT COMMITS")
print("=" * 65)

run_git("branch", "--show-current")
run_git("log", "-5", "--oneline")


# 6. Inspect the repository structure
print("\n" + "=" * 65)
print("4. REPOSITORY STRUCTURE")
print("=" * 65)

for item in sorted(
    REPO_ROOT.iterdir(),
    key=lambda p: (p.is_file(), p.name.lower())
):
    if item.name == ".git":
        continue

    suffix = "/" if item.is_dir() else ""
    print(f"{item.name}{suffix}")


# 7. Inspect project files up to three directory levels
print("\n" + "=" * 65)
print("5. PROJECT FILES")
print("=" * 65)

EXCLUDED_DIRS = {
    ".git",
    "__pycache__",
    ".ipynb_checkpoints"
}

for path in sorted(REPO_ROOT.rglob("*")):
    relative_path = path.relative_to(REPO_ROOT)

    if any(part in EXCLUDED_DIRS for part in relative_path.parts):
        continue

    if len(relative_path.parts) > 3:
        continue

    if path.is_file():
        print(relative_path)


# 8. Inspect SQL analytics files
print("\n" + "=" * 65)
print("6. SQL ANALYTICS FILES")
print("=" * 65)

SQL_DIR = REPO_ROOT / "sql"

if SQL_DIR.is_dir():
    sql_files = sorted(SQL_DIR.glob("*.sql"))

    for sql_file in sql_files:
        print(sql_file.name)

    print(f"\nTotal SQL files found: {len(sql_files)}")
else:
    print("SQL directory does not exist.")


# 9. Inspect recent commit summaries
print("\n" + "=" * 65)
print("7. RECENT COMMIT DETAILS")
print("=" * 65)

run_git("log", "-3", "--stat", "--oneline")


print("\n" + "=" * 65)
print("REPOSITORY INSPECTION FINISHED")
print("No project files were modified by this inspection cell.")
print("=" * 65)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

1. PROJECT PATH VERIFICATION
Repository exists: True
Raw data directory exists: True
Processed data directory exists: True
Source directory exists: True

2. GIT STATUS

$ git status --short --branch
## main...origin/main [ahead 1]
 M 02_postgresql_data_warehouse.ipynb
?? 03_PySpark_transformation.ipynb


3. BRANCH AND RECENT COMMITS

$ git branch --show-current
main


$ git log -5 --oneline
a99d9c4 Add data ingestion and PostgreSQL warehouse notebooks
eb48739 Add data ingestion and PostgreSQL warehouse notebooks
b825e5f Integrate Neon PostgreSQL data warehouse
a4b6dba Clean up data ingestion notebook
6e249bb Build sales analytics data engineering pipeline


4. REPOSITORY STRUCTURE
config/
data/
docs/
logs/
notebooks/
reports/
src/
tests/
.gitignore
00_phase0_setup.ipynb
01_data_ingestion.ipynb
02_postgresql_data_warehouse.ipynb
03_PySpark_transformation.ipyn

Inspect Transformation and Data Schemas

In [12]:


from pathlib import Path
import ast
import pandas as pd

# Reuse paths established in Step 1
PACKAGE_DIR = SRC_DIR / "sales_pipeline"
TEST_FILE = REPO_ROOT / "tests" / "test_pipeline.py"

# ------------------------------------------------------------
# 1. Inspect existing Python modules and function signatures
# ------------------------------------------------------------
print("=" * 75)
print("1. EXISTING PIPELINE MODULES AND FUNCTIONS")
print("=" * 75)

for file_path in sorted(PACKAGE_DIR.glob("*.py")):
    print(f"\n--- {file_path.relative_to(REPO_ROOT)} ---")

    try:
        source = file_path.read_text(encoding="utf-8")
        tree = ast.parse(source)

        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                args = [arg.arg for arg in node.args.args]
                print(
                    f"  {'async ' if isinstance(node, ast.AsyncFunctionDef) else ''}"
                    f"{node.name}({', '.join(args)})"
                )
            elif isinstance(node, ast.ClassDef):
                print(f"  class {node.name}")

    except Exception as exc:
        print("Could not inspect module:", exc)


# ------------------------------------------------------------
# 2. Read the main transformation implementations
# ------------------------------------------------------------
MODULES_TO_REVIEW = [
    "transform.py",
    "transformation.py",
    "transform_sales.py",
    "transform_customers.py",
    "transform_products.py",
    "validate.py",
]

print("\n" + "=" * 75)
print("2. EXISTING TRANSFORMATION AND VALIDATION CODE")
print("=" * 75)

for filename in MODULES_TO_REVIEW:
    file_path = PACKAGE_DIR / filename

    print(f"\n{'=' * 65}")
    print(f"FILE: {filename}")
    print("=" * 65)

    if not file_path.exists():
        print("File not found.")
        continue

    source = file_path.read_text(encoding="utf-8")
    print(source[:15000])

    if len(source) > 15000:
        print("\n[Output truncated. Remaining characters:", len(source) - 15000, "]")


# ------------------------------------------------------------
# 3. Inspect processed analytics CSV schemas and sample records
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("3. PROCESSED ANALYTICS CSV SCHEMAS")
print("=" * 75)

csv_files = sorted(PROCESSED_DIR.glob("*.csv"))

if not csv_files:
    print("No processed CSV files found.")
else:
    for csv_file in csv_files:
        print(f"\n--- {csv_file.name} ---")

        try:
            df = pd.read_csv(csv_file, nrows=5)

            print("Columns:", df.columns.tolist())
            print("Sample dtypes:")
            print(df.dtypes.to_string())
            print("Sample rows:")
            print(df.head(3).to_string(index=False))

            # Count all records without loading the entire file
            with csv_file.open("r", encoding="utf-8-sig") as f:
                row_count = sum(1 for _ in f) - 1

            print("Total data rows:", row_count)

        except Exception as exc:
            print("CSV inspection error:", exc)


# ------------------------------------------------------------
# 4. Inspect available raw data files
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("4. RAW OLIST DATA FILES")
print("=" * 75)

if RAW_DIR.exists():
    for file_path in sorted(RAW_DIR.iterdir()):
        if file_path.is_file():
            print(
                f"{file_path.name} "
                f"({file_path.stat().st_size / (1024 * 1024):.2f} MB)"
            )
else:
    print("Raw data directory not found.")


# ------------------------------------------------------------
# 5. Inspect existing test cases
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("5. EXISTING TEST CASES")
print("=" * 75)

if TEST_FILE.exists():
    test_source = TEST_FILE.read_text(encoding="utf-8")
    test_tree = ast.parse(test_source)

    for node in ast.walk(test_tree):
        if isinstance(node, ast.FunctionDef) and node.name.startswith("test_"):
            print(f"  {node.name}")
else:
    print("Test file not found:", TEST_FILE)


# ------------------------------------------------------------
# 6. Inspect declared dependencies
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("6. REQUIREMENTS")
print("=" * 75)

requirements_file = REPO_ROOT / "requirements.txt"

if requirements_file.exists():
    print(requirements_file.read_text(encoding="utf-8"))
else:
    print("requirements.txt not found.")

print("\n" + "=" * 75)
print("INSPECTION COMPLETE — NO FILES MODIFIED")
print("=" * 75)


1. EXISTING PIPELINE MODULES AND FUNCTIONS

--- src/sales_pipeline/__init__.py ---

--- src/sales_pipeline/database.py ---
  create_tables(database_url)
  load_processed_csvs(database_url, processed_dir)

--- src/sales_pipeline/extract.py ---
  load_csv(data_dir, dataset_name, chunksize)

--- src/sales_pipeline/ingestion.py ---

--- src/sales_pipeline/pipeline.py ---
  run_pipeline(data_dir, output_dir, database_url)

--- src/sales_pipeline/transform.py ---
  build_order_analytics(data_dir)
  save_order_analytics(analytics, output_dir)

--- src/sales_pipeline/transform_customers.py ---
  build_customer_analytics(data_dir)
  save_customer_analytics(customer_analytics, output_dir)

--- src/sales_pipeline/transform_products.py ---
  build_product_analytics(data_dir)
  save_product_analytics(product_analytics, category_analytics, output_dir)

--- src/sales_pipeline/transform_sales.py ---
  build_sales_analytics(data_dir)
  save_sales_analytics(monthly_sales, status_summary, output_dir)

--

Inspect the CSV loader and notebook state

In [13]:


# 1. Inspect how raw CSV files are loaded
loader_file = SRC_DIR / "sales_pipeline" / "extract.py"

print("=" * 70)
print("1. EXISTING CSV LOADER")
print("=" * 70)

if loader_file.exists():
    print(loader_file.read_text(encoding="utf-8"))
else:
    print("Loader not found:", loader_file)


# 2. Inspect the current Spark notebook's cell structure
import json

spark_notebook = REPO_ROOT / "03_PySpark_transformation.ipynb"

print("\n" + "=" * 70)
print("2. EXISTING SPARK NOTEBOOK")
print("=" * 70)

if spark_notebook.exists():
    notebook_data = json.loads(
        spark_notebook.read_text(encoding="utf-8")
    )

    print("Notebook format:", notebook_data.get("nbformat"))
    print("Number of cells:", len(notebook_data.get("cells", [])))

    for index, cell in enumerate(notebook_data.get("cells", [])):
        source = cell.get("source", [])
        source_text = "".join(source) if isinstance(source, list) else source

        print(f"\nCell {index + 1} | Type: {cell.get('cell_type')}")
        print(source_text[:1200] or "(empty cell)")

        if len(source_text) > 1200:
            print("[Cell output truncated]")
else:
    print("Spark notebook not found.")


# 3. Inspect source CSV headers
print("\n" + "=" * 70)
print("3. SOURCE DATA HEADERS")
print("=" * 70)

import csv

source_files = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
}

for dataset_name, filename in source_files.items():
    file_path = RAW_DIR / filename

    print(f"\n{dataset_name}: {filename}")

    if file_path.exists():
        with file_path.open(
            "r", encoding="utf-8-sig", newline=""
        ) as file:
            reader = csv.reader(file)
            header = next(reader, [])
            sample = next(reader, [])

        print("Columns:", header)
        print("First row:", sample)
    else:
        print("File not found:", file_path)


print("\nInspection complete. No files modified.")


1. EXISTING CSV LOADER
from pathlib import Path
import pandas as pd


# Explicit mapping to the actual source filenames.
DATASET_FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}


# Required source columns, based on our profiling results.
DATASET_SCHEMAS = {
    "orders": [
        "order_id", "customer_id", "order_status",
        "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": [
        "order_id", "order_item_id", "product_id"

# Setup and varify the Apachr Spark

In [14]:


import os
import sys
import subprocess
import importlib.util

# 1. Check Java availability
print("=" * 65)
print("1. JAVA ENVIRONMENT")
print("=" * 65)

java_result = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

print(java_result.stderr or java_result.stdout)

if java_result.returncode != 0:
    raise RuntimeError(
        "Java is unavailable. We must resolve the Java environment "
        "before initializing Spark."
    )


# 2. Check whether PySpark is already installed
print("\n" + "=" * 65)
print("2. PYSPARK AVAILABILITY")
print("=" * 65)

pyspark_installed = importlib.util.find_spec("pyspark") is not None
print("PySpark installed:", pyspark_installed)

# Install only if it is missing; do not change existing versions unnecessarily.
if not pyspark_installed:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "pyspark"
    ])


# 3. Import PySpark and create a Spark session
print("\n" + "=" * 65)
print("3. SPARK SESSION")
print("=" * 65)

import pyspark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SalesAnalyticsPhase3")
    .getOrCreate()
)

print("PySpark version:", pyspark.__version__)
print("Spark version:", spark.version)
print("Application name:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)


# 4. Configure readable output for development
spark.conf.set("spark.sql.shuffle.partitions", "4")

print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)


# 5. Run a small execution test
print("\n" + "=" * 65)
print("4. SPARK EXECUTION TEST")
print("=" * 65)

test_df = spark.createDataFrame(
    [
        ("order_a", 100.0),
        ("order_b", 200.0),
        ("order_c", 150.0),
    ],
    ["order_id", "revenue"]
)

test_df.groupBy().sum("revenue").show()

total_revenue = test_df.agg({"revenue": "sum"}).first()[0]

assert total_revenue == 450.0, (
    f"Unexpected Spark result: {total_revenue}"
)

print("Spark execution test: PASS")
print("Total test revenue:", total_revenue)


1. JAVA ENVIRONMENT
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


2. PYSPARK AVAILABILITY
PySpark installed: True

3. SPARK SESSION
PySpark version: 4.0.4
Spark version: 4.0.4
Application name: SalesAnalyticsPhase3
Master: local[*]
Shuffle partitions: 4

4. SPARK EXECUTION TEST
+------------+
|sum(revenue)|
+------------+
|       450.0|
+------------+

Spark execution test: PASS
Total test revenue: 450.0


Explicit Schemas and csv investigation

In [15]:



from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType,
)

# ------------------------------------------------------------
# 1. Define explicit schemas
# ------------------------------------------------------------

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True),
])

order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True),
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),
])

# ------------------------------------------------------------
# 2. Read CSV files using the defined schemas
# ------------------------------------------------------------

orders_path = str(RAW_DIR / "olist_orders_dataset.csv")
items_path = str(RAW_DIR / "olist_order_items_dataset.csv")
customers_path = str(RAW_DIR / "olist_customers_dataset.csv")

orders_df = (
    spark.read
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .schema(orders_schema)
    .csv(orders_path)
)

order_items_df = (
    spark.read
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .schema(order_items_schema)
    .csv(items_path)
)

customers_df = (
    spark.read
    .option("header", True)
    .schema(customers_schema)
    .csv(customers_path)
)

# ------------------------------------------------------------
# 3. Verify schemas and sample records
# ------------------------------------------------------------

for name, df in [
    ("ORDERS", orders_df),
    ("ORDER ITEMS", order_items_df),
    ("CUSTOMERS", customers_df),
]:
    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

    df.printSchema()
    df.show(5, truncate=False)

# ------------------------------------------------------------
# 4. Validate required columns and data types
# ------------------------------------------------------------

assert orders_df.columns == orders_schema.fieldNames()
assert order_items_df.columns == order_items_schema.fieldNames()
assert customers_df.columns == customers_schema.fieldNames()

assert isinstance(
    orders_df.schema["order_purchase_timestamp"].dataType,
    TimestampType,
)

assert isinstance(
    order_items_df.schema["order_item_id"].dataType,
    IntegerType,
)

assert isinstance(
    order_items_df.schema["price"].dataType,
    DoubleType,
)

print("\nExplicit schema checks: PASS")



ORDERS
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+--

Clean and validate source DataFrames

In [16]:

from pyspark.sql import functions as F

# Keep the original DataFrames unchanged.
orders_raw_df = orders_df
order_items_raw_df = order_items_df
customers_raw_df = customers_df


# ------------------------------------------------------------
# 1. Standardize string columns
# ------------------------------------------------------------

def standardize_strings(df, lowercase_columns=None):
    """Trim string values and optionally lowercase selected columns."""
    lowercase_columns = set(lowercase_columns or [])

    for field in df.schema.fields:
        if field.dataType.simpleString() == "string":
            column = F.trim(F.col(field.name))

            if field.name in lowercase_columns:
                column = F.lower(column)

            df = df.withColumn(field.name, column)

    return df


orders_clean_df = standardize_strings(
    orders_raw_df,
    lowercase_columns=["order_status"],
)

order_items_clean_df = standardize_strings(order_items_raw_df)
customers_clean_df = standardize_strings(customers_raw_df)


# ------------------------------------------------------------
# 2. Define a reusable quality profiling function
# ------------------------------------------------------------

def profile_dataframe(df, name, key_columns):
    print("\n" + "=" * 70)
    print(f"DATA QUALITY PROFILE: {name}")
    print("=" * 70)

    row_count = df.count()
    print("Total rows:", row_count)

    # Null or blank key values
    invalid_key_condition = None

    for column in key_columns:
        condition = (
            F.col(column).isNull()
            | (F.trim(F.col(column)) == "")
        )

        invalid_key_condition = (
            condition
            if invalid_key_condition is None
            else invalid_key_condition | condition
        )

    invalid_key_count = df.filter(invalid_key_condition).count()
    print("Rows with null/blank key fields:", invalid_key_count)

    # Duplicate business keys
    duplicate_key_count = (
        df.groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print("Duplicate key groups:", duplicate_key_count)

    return {
        "dataset": name,
        "rows": row_count,
        "invalid_key_rows": invalid_key_count,
        "duplicate_key_groups": duplicate_key_count,
    }


# ------------------------------------------------------------
# 3. Profile primary keys and composite keys
# ------------------------------------------------------------

quality_results = []

quality_results.append(
    profile_dataframe(
        orders_clean_df,
        "orders",
        ["order_id"],
    )
)

quality_results.append(
    profile_dataframe(
        order_items_clean_df,
        "order_items",
        ["order_id", "order_item_id"],
    )
)

quality_results.append(
    profile_dataframe(
        customers_clean_df,
        "customers",
        ["customer_id"],
    )
)


# ------------------------------------------------------------
# 4. Check invalid numeric values in order items
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ORDER ITEM NUMERIC VALIDATION")
print("=" * 70)

numeric_checks = {
    "Null order_item_id": F.col("order_item_id").isNull(),
    "Non-positive order_item_id": F.col("order_item_id") <= 0,
    "Null price": F.col("price").isNull(),
    "Negative price": F.col("price") < 0,
    "Null freight_value": F.col("freight_value").isNull(),
    "Negative freight_value": F.col("freight_value") < 0,
}

for check_name, condition in numeric_checks.items():
    count = order_items_clean_df.filter(condition).count()
    print(f"{check_name}: {count}")


# ------------------------------------------------------------
# 5. Check customer and relationship keys
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RELATIONSHIP KEY VALIDATION")
print("=" * 70)

relationship_checks = [
    (
        "Orders with missing customer_id",
        orders_clean_df.filter(
            F.col("customer_id").isNull()
            | (F.col("customer_id") == "")
        ),
    ),
    (
        "Customers with missing customer_unique_id",
        customers_clean_df.filter(
            F.col("customer_unique_id").isNull()
            | (F.col("customer_unique_id") == "")
        ),
    ),
    (
        "Order items with missing product_id",
        order_items_clean_df.filter(
            F.col("product_id").isNull()
            | (F.col("product_id") == "")
        ),
    ),
    (
        "Order items with missing seller_id",
        order_items_clean_df.filter(
            F.col("seller_id").isNull()
            | (F.col("seller_id") == "")
        ),
    ),
]

for check_name, invalid_df in relationship_checks:
    print(f"{check_name}: {invalid_df.count()}")


# ------------------------------------------------------------
# 6. Check referential integrity without dropping records
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("REFERENTIAL INTEGRITY")
print("=" * 70)

order_ids = orders_clean_df.select("order_id").distinct()
customer_ids = customers_clean_df.select("customer_id").distinct()

orphan_items = order_items_clean_df.join(
    order_ids,
    on="order_id",
    how="left_anti",
)

orphan_orders = orders_clean_df.join(
    customer_ids,
    on="customer_id",
    how="left_anti",
)

print("Order items referencing missing orders:", orphan_items.count())
print("Orders referencing missing customers:", orphan_orders.count())


# ------------------------------------------------------------
# 7. Display a compact summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("QUALITY SUMMARY")
print("=" * 70)

quality_summary_df = spark.createDataFrame(quality_results)
quality_summary_df.show(truncate=False)

print("Profiling completed. No records have been intentionally dropped.")



DATA QUALITY PROFILE: orders
Total rows: 99441
Rows with null/blank key fields: 0
Duplicate key groups: 0

DATA QUALITY PROFILE: order_items
Total rows: 112650
Rows with null/blank key fields: 0
Duplicate key groups: 0

DATA QUALITY PROFILE: customers
Total rows: 99441
Rows with null/blank key fields: 0
Duplicate key groups: 0

ORDER ITEM NUMERIC VALIDATION
Null order_item_id: 0
Non-positive order_item_id: 0
Null price: 0
Negative price: 0
Null freight_value: 0
Negative freight_value: 0

RELATIONSHIP KEY VALIDATION
Orders with missing customer_id: 0
Customers with missing customer_unique_id: 0
Order items with missing product_id: 0
Order items with missing seller_id: 0

REFERENTIAL INTEGRITY
Order items referencing missing orders: 0
Orders referencing missing customers: 0

QUALITY SUMMARY
+-----------+--------------------+----------------+------+
|dataset    |duplicate_key_groups|invalid_key_rows|rows  |
+-----------+--------------------+----------------+------+
|orders     |0        

# Design the Spark Tranformation

Build order-level revenue metrics

In [17]:

from pyspark.sql import functions as F


#  PYSPARK ORDER-LEVEL TRANSFORMATION


# 1. Aggregate order items to one row per order.
item_metrics_df = order_items_clean_df.groupBy("order_id").agg(
    F.count("order_item_id").alias("item_count"),
    F.countDistinct("product_id").alias("unique_products"),
    F.countDistinct("seller_id").alias("unique_sellers"),
    F.sum("price").alias("item_revenue"),
    F.sum("freight_value").alias("freight_cost"),
)

# 2. Join item metrics to orders.
order_sales_df = (
    orders_clean_df
    .join(item_metrics_df, on="order_id", how="left")
    .fillna(
        0,
        subset=["item_count", "unique_products", "unique_sellers"]
    )
)

# 3. Add customer attributes.
customer_attributes_df = customers_clean_df.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
)

order_sales_df = order_sales_df.join(
    customer_attributes_df,
    on="customer_id",
    how="left",
)

# 4. Derive delivery duration metrics.
seconds_per_day = 86400.0

order_sales_df = (
    order_sales_df
    .withColumn(
        "delivery_duration_days",
        (
            F.col("order_delivered_customer_date").cast("long")
            - F.col("order_purchase_timestamp").cast("long")
        ) / F.lit(seconds_per_day),
    )
    .withColumn(
        "estimated_delivery_duration_days",
        (
            F.col("order_estimated_delivery_date").cast("long")
            - F.col("order_purchase_timestamp").cast("long")
        ) / F.lit(seconds_per_day),
    )
    .withColumn(
        "delivery_delay_days",
        (
            F.col("order_delivered_customer_date").cast("long")
            - F.col("order_estimated_delivery_date").cast("long")
        ) / F.lit(seconds_per_day),
    )
)

# 5. Derive delivery flags.
order_sales_df = (
    order_sales_df
    .withColumn(
        "is_delivered",
        F.col("order_status") == F.lit("delivered"),
    )
    .withColumn(
        "is_late_delivery",
        F.when(
            F.col("order_status") == F.lit("delivered"),
            F.coalesce(
                F.col("delivery_delay_days") > F.lit(0.0),
                F.lit(False),
            ),
        ).otherwise(F.lit(False)),
    )
)

# 6. Validate the transformation.
source_order_count = orders_clean_df.count()
output_order_count = order_sales_df.count()

duplicate_order_ids = (
    order_sales_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_late_flags = order_sales_df.filter(
    F.col("is_late_delivery").isNull()
).count()

print("=" * 55)
print("ORDER-LEVEL TRANSFORMATION VALIDATION")
print("=" * 55)
print("Source orders:", source_order_count)
print("Output orders:", output_order_count)
print("Duplicate order IDs:", duplicate_order_ids)
print("NULL is_late_delivery flags:", null_late_flags)

if (
    source_order_count == output_order_count
    and duplicate_order_ids == 0
    and null_late_flags == 0
):
    print("\nPASS: Order-level transformation checks passed.")
else:
    raise ValueError(
        "Transformation validation failed. "
        "Inspect row counts, duplicate IDs, and delivery flags."
    )

# 7. Preview the transformed data.
order_sales_df.select(
    "order_id",
    "order_status",
    "item_count",
    "item_revenue",
    "freight_cost",
    "delivery_delay_days",
    "is_delivered",
    "is_late_delivery",
).show(10, truncate=False)


ORDER-LEVEL TRANSFORMATION VALIDATION
Source orders: 99441
Output orders: 99441
Duplicate order IDs: 0
NULL is_late_delivery flags: 0

PASS: Order-level transformation checks passed.
+--------------------------------+------------+----------+------------+------------+-------------------+------------+----------------+
|order_id                        |order_status|item_count|item_revenue|freight_cost|delivery_delay_days|is_delivered|is_late_delivery|
+--------------------------------+------------+----------+------------+------------+-------------------+------------+----------------+
|e481f51cbdc54678b7cc49136f2d6af7|delivered   |1         |29.99       |8.72        |-7.107488425925926 |true        |false           |
|53cdb2fc8bc7dce0b6741e2150273451|delivered   |1         |118.7       |22.76       |-5.355729166666666 |true        |false           |
|47770eb9100c2d0c44946d9cf07ec65d|delivered   |1         |159.9       |19.22       |-17.245497685185185|true        |false           |
|949d5b

Compare Spark vs Python Reconciliation

In [18]:

from pyspark.sql import functions as F


# STEP 8: RECONCILE PYSPARK OUTPUT WITH PYTHON REFERENCE


reference_path = str(PROCESSED_DIR / "order_analytics.csv")

# 1. Load Python reference output
reference_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(reference_path)
)

print("Reference row count:", reference_df.count())
print("Spark output row count:", order_sales_df.count())

# 2. Define shared metrics
numeric_metrics = [
    "item_count",
    "unique_products",
    "unique_sellers",
    "item_revenue",
    "freight_cost",
    "delivery_duration_days",
    "estimated_delivery_duration_days",
    "delivery_delay_days",
]

boolean_metrics = [
    "is_delivered",
    "is_late_delivery",
]

required_columns = ["order_id"] + numeric_metrics + boolean_metrics

# Verify schemas contain the required columns
missing_reference = [
    c for c in required_columns if c not in reference_df.columns
]
missing_spark = [
    c for c in required_columns if c not in order_sales_df.columns
]

if missing_reference or missing_spark:
    raise ValueError(
        f"Missing reference columns: {missing_reference}; "
        f"Missing Spark columns: {missing_spark}"
    )

# 3. Standardize types
reference_compare_df = reference_df.select(
    "order_id",
    *[
        F.col(c).cast("double").alias(c)
        for c in numeric_metrics
    ],
    *[
        F.lower(F.trim(F.col(c))).cast("boolean").alias(c)
        for c in boolean_metrics
    ],
)

spark_compare_df = order_sales_df.select(
    "order_id",
    *[
        F.col(c).cast("double").alias(c)
        for c in numeric_metrics
    ],
    *[
        F.col(c).cast("boolean").alias(c)
        for c in boolean_metrics
    ],
)

# 4. Duplicate-key checks
reference_duplicates = (
    reference_compare_df.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

spark_duplicates = (
    spark_compare_df.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 5. Key reconciliation
reference_keys = reference_compare_df.select("order_id").distinct()
spark_keys = spark_compare_df.select("order_id").distinct()

missing_in_spark = reference_keys.join(
    spark_keys, "order_id", "left_anti"
).count()

extra_in_spark = spark_keys.join(
    reference_keys, "order_id", "left_anti"
).count()

print("\nKEY RECONCILIATION")
print("Reference duplicate order IDs:", reference_duplicates)
print("Spark duplicate order IDs:", spark_duplicates)
print("Reference orders missing from Spark:", missing_in_spark)
print("Extra Spark orders:", extra_in_spark)

# 6. Join datasets for comparison
joined_df = reference_compare_df.alias("ref").join(
    spark_compare_df.alias("spk"),
    on="order_id",
    how="inner",
)

tolerance = 1e-6
metric_results = {}

print("\nMETRIC RECONCILIATION")

# Numeric metrics: null-safe comparison with tolerance
for metric in numeric_metrics:
    ref_value = F.col(f"ref.{metric}")
    spark_value = F.col(f"spk.{metric}")

    matches = (
        ref_value.eqNullSafe(spark_value)
        | (
            ref_value.isNotNull()
            & spark_value.isNotNull()
            & (F.abs(ref_value - spark_value) <= tolerance)
        )
    )

    mismatch_count = joined_df.filter(~matches).count()
    metric_results[metric] = mismatch_count == 0

    status = "PASS" if mismatch_count == 0 else "REVIEW"
    print(f"{status}: {metric} — {mismatch_count} mismatched rows")

# Boolean metrics: null-safe exact comparison
for metric in boolean_metrics:
    matches = F.col(f"ref.{metric}").eqNullSafe(
        F.col(f"spk.{metric}")
    )

    mismatch_count = joined_df.filter(~matches).count()
    metric_results[metric] = mismatch_count == 0

    status = "PASS" if mismatch_count == 0 else "REVIEW"
    print(f"{status}: {metric} — {mismatch_count} mismatched rows")

# 7. Overall result
reference_count = reference_compare_df.count()
spark_count = spark_compare_df.count()

row_counts_match = reference_count == spark_count
keys_match = missing_in_spark == 0 and extra_in_spark == 0
unique_keys = reference_duplicates == 0 and spark_duplicates == 0
all_metrics_match = all(metric_results.values())

reconciliation_passed = (
    row_counts_match
    and keys_match
    and unique_keys
    and all_metrics_match
)

print("\n" + "=" * 55)
print("STEP 8: RECONCILIATION SUMMARY")
print("=" * 55)
print("Reference rows:", reference_count)
print("Spark rows:", spark_count)
print("Row counts match:", row_counts_match)
print("Keys match:", keys_match)
print("Keys are unique:", unique_keys)
print("All compared metrics match:", all_metrics_match)
print("RECONCILIATION PASSED:", reconciliation_passed)
print("=" * 55)

if not reconciliation_passed:
    print("REVIEW REQUIRED: Investigate the failed checks above.")
else:
    print("SUCCESS: All selected metrics match the Python reference.")


Reference row count: 99441
Spark output row count: 99441

KEY RECONCILIATION
Reference duplicate order IDs: 0
Spark duplicate order IDs: 0
Reference orders missing from Spark: 0
Extra Spark orders: 0

METRIC RECONCILIATION
PASS: item_count — 0 mismatched rows
PASS: unique_products — 0 mismatched rows
PASS: unique_sellers — 0 mismatched rows
PASS: item_revenue — 0 mismatched rows
PASS: freight_cost — 0 mismatched rows
PASS: delivery_duration_days — 0 mismatched rows
PASS: estimated_delivery_duration_days — 0 mismatched rows
PASS: delivery_delay_days — 0 mismatched rows
PASS: is_delivered — 0 mismatched rows
PASS: is_late_delivery — 0 mismatched rows

STEP 8: RECONCILIATION SUMMARY
Reference rows: 99441
Spark rows: 99441
Row counts match: True
Keys match: True
Keys are unique: True
All compared metrics match: True
RECONCILIATION PASSED: True
SUCCESS: All selected metrics match the Python reference.


In [19]:
from pyspark.sql import functions as F

order_sales_df.filter(
    F.col("order_id").isin([
        "2d1e2d5bf4dc7227b3bfebb81328c15f",
        "f5dd62b788049ad9fc0526e3ad11a097",
        "2ebdfc4f15f23b91474edf87475f108e",
        "e69f75a717d64fc5ecdfae42b2e8e086",
        "0d3268bad9b086af767785e3f0fc0133",
        "2d858f451373b04fb5c984a1cc2defaf",
        "ab7c89dc1bf4a1ead9d6ec1ec8968a84",
        "20edc82cf5400ce95e1afacc25798b31",
    ])
).select(
    "order_id",
    "delivery_delay_days",
    "is_delivered",
    "is_late_delivery",
).show(truncate=False)

+--------------------------------+-------------------+------------+----------------+
|order_id                        |delivery_delay_days|is_delivered|is_late_delivery|
+--------------------------------+-------------------+------------+----------------+
|2d1e2d5bf4dc7227b3bfebb81328c15f|NULL               |true        |false           |
|f5dd62b788049ad9fc0526e3ad11a097|NULL               |true        |false           |
|2ebdfc4f15f23b91474edf87475f108e|NULL               |true        |false           |
|e69f75a717d64fc5ecdfae42b2e8e086|NULL               |true        |false           |
|0d3268bad9b086af767785e3f0fc0133|NULL               |true        |false           |
|2d858f451373b04fb5c984a1cc2defaf|NULL               |true        |false           |
|ab7c89dc1bf4a1ead9d6ec1ec8968a84|NULL               |true        |false           |
|20edc82cf5400ce95e1afacc25798b31|NULL               |true        |false           |
+--------------------------------+-------------------+-----------

# Payment aggregation

Load the payment dataset with an explicit schema

In [20]:

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
)



payments_path = RAW_DIR / "olist_order_payments_dataset.csv"

payments_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("payment_sequential", IntegerType(), True),
    StructField("payment_type", StringType(), True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value", DoubleType(), True),
])

payments_raw_df = (
    spark.read
    .option("header", True)
    .schema(payments_schema)
    .csv(str(payments_path))
)

# Standardize string values.
payments_clean_df = (
    payments_raw_df
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("payment_type", F.lower(F.trim(F.col("payment_type"))))
)

# ============================================================
# VALIDATION
# ============================================================

print("=" * 55)
print("PAYMENT DATASET VALIDATION")
print("=" * 55)

print("Source file:", payments_path.name)
print("Payment records:", payments_clean_df.count())
print("Schema:")
payments_clean_df.printSchema()

print("\nNull counts:")
payments_clean_df.select(
    *[
        F.sum(F.col(c).isNull().cast("long")).alias(c)
        for c in payments_clean_df.columns
    ]
).show(truncate=False)

print("Distinct orders:", payments_clean_df.select("order_id").distinct().count())

print("\nSample payment records:")
payments_clean_df.show(10, truncate=False)




PAYMENT DATASET VALIDATION
Source file: olist_order_payments_dataset.csv
Payment records: 103886
Schema:
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)


Null counts:
+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|0       |0                 |0           |0                   |0            |
+--------+------------------+------------+--------------------+-------------+

Distinct orders: 99440

Sample payment records:
+--------------------------------+------------------+------------+--------------------+-------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|

Aggregate payments and join to orders

In [21]:

from pyspark.sql import functions as F

# 1. Aggregate payment records by order.
payment_metrics_df = payments_clean_df.groupBy("order_id").agg(
    F.sum("payment_value").alias("payment_total"),
    F.count("payment_sequential").alias("payment_records"),
    F.countDistinct("payment_type").alias("payment_types"),
)

# 2. Join payment metrics to the existing order-level DataFrame.
order_sales_df = order_sales_df.join(
    payment_metrics_df,
    on="order_id",
    how="left",
)

# 3. Validate the join.
source_order_count = orders_clean_df.count()
output_order_count = order_sales_df.count()

duplicate_order_ids = (
    order_sales_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

missing_payment_metrics = order_sales_df.filter(
    F.col("payment_total").isNull()
).count()

print("=" * 55)
print("PAYMENT AGGREGATION VALIDATION")
print("=" * 55)
print("Payment source records:", payments_clean_df.count())
print("Distinct payment orders:", payment_metrics_df.count())
print("Source orders:", source_order_count)
print("Output orders:", output_order_count)
print("Duplicate order IDs:", duplicate_order_ids)
print("Orders without payment metrics:", missing_payment_metrics)

if (
    source_order_count == output_order_count
    and duplicate_order_ids == 0
):
    print("\nPASS: Payment aggregation preserved the order grain.")
else:
    raise ValueError(
        "Payment aggregation validation failed. "
        "Check row counts and duplicate order IDs."
    )

# 4. Preview payment metrics.
order_sales_df.select(
    "order_id",
    "item_revenue",
    "freight_cost",
    "payment_total",
    "payment_records",
    "payment_types",
).show(10, truncate=False)




PAYMENT AGGREGATION VALIDATION
Payment source records: 103886
Distinct payment orders: 99440
Source orders: 99441
Output orders: 99441
Duplicate order IDs: 0
Orders without payment metrics: 1

PASS: Payment aggregation preserved the order grain.
+--------------------------------+------------+------------+-------------+---------------+-------------+
|order_id                        |item_revenue|freight_cost|payment_total|payment_records|payment_types|
+--------------------------------+------------+------------+-------------+---------------+-------------+
|e481f51cbdc54678b7cc49136f2d6af7|29.99       |8.72        |38.71        |3              |2            |
|53cdb2fc8bc7dce0b6741e2150273451|118.7       |22.76       |141.46       |1              |1            |
|47770eb9100c2d0c44946d9cf07ec65d|159.9       |19.22       |179.12       |1              |1            |
|949d5b44dbf5de918fe9c16f97b45f8a|45.0        |27.2        |72.2         |1              |1            |
|ad21c59c0840e6cb83

In [22]:

from pyspark.sql import functions as F

# ============================================================
# STANDARDIZE MISSING PAYMENT COUNTS
# ============================================================

# Preserve NULL payment_total when no payment record exists.
# Set payment_records and payment_types to zero for those orders.
order_sales_df = order_sales_df.withColumn(
    "payment_records",
    F.coalesce(F.col("payment_records"), F.lit(0)).cast("long"),
).withColumn(
    "payment_types",
    F.coalesce(F.col("payment_types"), F.lit(0)).cast("long"),
)

# Validate the previously mismatched order.
problem_order_id = "bfbd0f9bdef84302105ad712db648a6c"

order_sales_df.filter(
    F.col("order_id") == problem_order_id
).select(
    "order_id",
    "payment_total",
    "payment_records",
    "payment_types",
).show(truncate=False)

# Confirm no count metrics remain NULL.
null_payment_counts = order_sales_df.filter(
    F.col("payment_records").isNull()
    | F.col("payment_types").isNull()
).count()

print("Orders with NULL payment_records/payment_types:", null_payment_counts)

if null_payment_counts == 0:
    print("PASS: Payment count metrics are standardized.")
else:
    raise ValueError("Payment count metrics still contain NULL values.")

print("\n No files modified.")


+--------------------------------+-------------+---------------+-------------+
|order_id                        |payment_total|payment_records|payment_types|
+--------------------------------+-------------+---------------+-------------+
|bfbd0f9bdef84302105ad712db648a6c|NULL         |0              |0            |
+--------------------------------+-------------+---------------+-------------+

Orders with NULL payment_records/payment_types: 0
PASS: Payment count metrics are standardized.

 No files modified.


Update the reconciliation to include the payment metrics

In [23]:

from pyspark.sql import functions as F

# ============================================================
# FULL ORDER-LEVEL RECONCILIATION
# ============================================================

reference_path = str(PROCESSED_DIR / "order_analytics.csv")

reference_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(reference_path)
)

numeric_metrics = [
    "item_count",
    "unique_products",
    "unique_sellers",
    "item_revenue",
    "freight_cost",
    "payment_total",
    "payment_records",
    "payment_types",
    "delivery_duration_days",
    "estimated_delivery_duration_days",
    "delivery_delay_days",
]

boolean_metrics = [
    "is_delivered",
    "is_late_delivery",
]

required_columns = ["order_id"] + numeric_metrics + boolean_metrics

missing_reference = [
    c for c in required_columns if c not in reference_df.columns
]
missing_spark = [
    c for c in required_columns if c not in order_sales_df.columns
]

if missing_reference or missing_spark:
    raise ValueError(
        f"Missing reference columns: {missing_reference}; "
        f"Missing Spark columns: {missing_spark}"
    )

# Convert reference values to comparable types.
reference_compare_df = reference_df.select(
    "order_id",
    *[
        F.col(c).cast("double").alias(c)
        for c in numeric_metrics
    ],
    *[
        F.lower(F.trim(F.col(c))).cast("boolean").alias(c)
        for c in boolean_metrics
    ],
)

spark_compare_df = order_sales_df.select(
    "order_id",
    *[
        F.col(c).cast("double").alias(c)
        for c in numeric_metrics
    ],
    *[
        F.col(c).cast("boolean").alias(c)
        for c in boolean_metrics
    ],
)

# Check key uniqueness.
reference_duplicates = (
    reference_compare_df.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

spark_duplicates = (
    spark_compare_df.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Check missing and extra order IDs.
reference_keys = reference_compare_df.select("order_id").distinct()
spark_keys = spark_compare_df.select("order_id").distinct()

missing_in_spark = reference_keys.join(
    spark_keys, "order_id", "left_anti"
).count()

extra_in_spark = spark_keys.join(
    reference_keys, "order_id", "left_anti"
).count()

# Join on order_id for metric comparisons.
joined_df = reference_compare_df.alias("ref").join(
    spark_compare_df.alias("spk"),
    on="order_id",
    how="inner",
)

tolerance = 1e-6
metric_results = {}

print("=" * 60)
print("FULL METRIC RECONCILIATION")
print("=" * 60)

# Compare numeric metrics, allowing null to match null.
for metric in numeric_metrics:
    ref_value = F.col(f"ref.{metric}")
    spark_value = F.col(f"spk.{metric}")

    matches = (
        ref_value.eqNullSafe(spark_value)
        | (
            ref_value.isNotNull()
            & spark_value.isNotNull()
            & (F.abs(ref_value - spark_value) <= tolerance)
        )
    )

    mismatch_count = joined_df.filter(~matches).count()
    metric_results[metric] = mismatch_count == 0

    status = "PASS" if mismatch_count == 0 else "REVIEW"
    print(f"{status}: {metric} — {mismatch_count} mismatched rows")

# Compare Boolean metrics exactly, with null-safe equality.
for metric in boolean_metrics:
    matches = F.col(f"ref.{metric}").eqNullSafe(
        F.col(f"spk.{metric}")
    )

    mismatch_count = joined_df.filter(~matches).count()
    metric_results[metric] = mismatch_count == 0

    status = "PASS" if mismatch_count == 0 else "REVIEW"
    print(f"{status}: {metric} — {mismatch_count} mismatched rows")

# Final reconciliation checks.
reference_count = reference_compare_df.count()
spark_count = spark_compare_df.count()

row_counts_match = reference_count == spark_count
keys_match = missing_in_spark == 0 and extra_in_spark == 0
unique_keys = reference_duplicates == 0 and spark_duplicates == 0
all_metrics_match = all(metric_results.values())

reconciliation_passed = (
    row_counts_match
    and keys_match
    and unique_keys
    and all_metrics_match
)

print("\n" + "=" * 60)
print("RECONCILIATION SUMMARY")
print("=" * 60)
print("Reference rows:", reference_count)
print("Spark rows:", spark_count)
print("Reference duplicate IDs:", reference_duplicates)
print("Spark duplicate IDs:", spark_duplicates)
print("Missing reference orders in Spark:", missing_in_spark)
print("Extra Spark orders:", extra_in_spark)
print("Row counts match:", row_counts_match)
print("Keys match:", keys_match)
print("All 13 metrics match:", all_metrics_match)
print("RECONCILIATION PASSED:", reconciliation_passed)
print("=" * 60)

if not reconciliation_passed:
    print("REVIEW REQUIRED: Investigate any metric marked REVIEW.")
else:
    print("SUCCESS: All 13 metrics match the Python reference.")


FULL METRIC RECONCILIATION
PASS: item_count — 0 mismatched rows
PASS: unique_products — 0 mismatched rows
PASS: unique_sellers — 0 mismatched rows
PASS: item_revenue — 0 mismatched rows
PASS: freight_cost — 0 mismatched rows
PASS: payment_total — 0 mismatched rows
PASS: payment_records — 0 mismatched rows
PASS: payment_types — 0 mismatched rows
PASS: delivery_duration_days — 0 mismatched rows
PASS: estimated_delivery_duration_days — 0 mismatched rows
PASS: delivery_delay_days — 0 mismatched rows
PASS: is_delivered — 0 mismatched rows
PASS: is_late_delivery — 0 mismatched rows

RECONCILIATION SUMMARY
Reference rows: 99441
Spark rows: 99441
Reference duplicate IDs: 0
Spark duplicate IDs: 0
Missing reference orders in Spark: 0
Extra Spark orders: 0
Row counts match: True
Keys match: True
All 13 metrics match: True
RECONCILIATION PASSED: True
SUCCESS: All 13 metrics match the Python reference.


## Analyze partitioning and execution

INSPECT SPARK PARTITIONING AND EXECUTION PLAN


In [24]:

print("SPARK EXECUTION CONFIGURATION")

print("Spark version:", spark.version)
print("Execution mode:", spark.sparkContext.master)
print(
    "Configured shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

print("\n" + "=" * 60)
print("DATAFRAME PARTITIONING")
print("=" * 60)

print(
    "Order-level DataFrame partitions:",
    order_sales_df.rdd.getNumPartitions()
)

print(
    "Order-items DataFrame partitions:",
    order_items_clean_df.rdd.getNumPartitions()
)

print(
    "Payment DataFrame partitions:",
    payments_clean_df.rdd.getNumPartitions()
)

print("\n" + "=" * 60)
print("ORDER-LEVEL EXECUTION PLAN")
print("=" * 60)

order_sales_df.explain(mode="formatted")

print("\n No files modified.")


SPARK EXECUTION CONFIGURATION
Spark version: 4.0.4
Execution mode: local[*]
Configured shuffle partitions: 4

DATAFRAME PARTITIONING
Order-level DataFrame partitions: 2
Order-items DataFrame partitions: 2
Payment DataFrame partitions: 2

ORDER-LEVEL EXECUTION PLAN
== Physical Plan ==
AdaptiveSparkPlan (82)
+- == Final Plan ==
   ResultQueryStage (49)
   +- * Project (48)
      +- * BroadcastHashJoin LeftOuter BuildRight (47)
         :- * Project (31)
         :  +- * Project (30)
         :     +- * BroadcastHashJoin LeftOuter BuildRight (29)
         :        :- * Project (23)
         :        :  +- * SortMergeJoin LeftOuter (22)
         :        :     :- * Sort (6)
         :        :     :  +- AQEShuffleRead (5)
         :        :     :     +- ShuffleQueryStage (4), Statistics(sizeInBytes=14.4 MiB, rowCount=9.94E+4)
         :        :     :        +- Exchange (3)
         :        :     :           +- * Project (2)
         :        :     :              +- Scan csv  (1)
       

Measure execution time and partition distribution

In [25]:

import time
from pyspark.sql import functions as F


print("BASELINE SPARK EXECUTION BENCHMARK")

print("Input order-level partitions:", order_sales_df.rdd.getNumPartitions())

# Measure a full aggregation over the transformed DataFrame.
start_time = time.perf_counter()

baseline_metrics = order_sales_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("order_id").alias("distinct_orders"),
    F.sum("item_revenue").alias("total_item_revenue"),
    F.sum("freight_cost").alias("total_freight_cost"),
    F.sum("payment_total").alias("total_payment_value"),
).collect()[0]

baseline_seconds = time.perf_counter() - start_time

print("\nBaseline results:")
print("Rows:", baseline_metrics["row_count"])
print("Distinct orders:", baseline_metrics["distinct_orders"])
print("Total item revenue:", baseline_metrics["total_item_revenue"])
print("Total freight cost:", baseline_metrics["total_freight_cost"])
print("Total payment value:", baseline_metrics["total_payment_value"])
print(f"Execution time: {baseline_seconds:.3f} seconds")

# Inspect the row distribution across current partitions.
partition_distribution = (
    order_sales_df
    .withColumn(
        "_partition_id",
        F.spark_partition_id()
    )
    .groupBy("_partition_id")
    .count()
    .orderBy("_partition_id")
)

print("\nRows per output partition:")
partition_distribution.show(truncate=False)

print("\n No files modified.")


BASELINE SPARK EXECUTION BENCHMARK
Input order-level partitions: 2

Baseline results:
Rows: 99441
Distinct orders: 99441
Total item revenue: 13591643.70000265
Total freight cost: 2251909.5399999423
Total payment value: 16008872.119999733
Execution time: 4.867 seconds

Rows per output partition:
+-------------+-----+
|_partition_id|count|
+-------------+-----+
|0            |61530|
|1            |37911|
+-------------+-----+


 No files modified.


Test partitioning

In [26]:
import time
from pyspark.sql import functions as F

print("=" * 60)
print("REPARTITIONED SPARK EXECUTION BENCHMARK")
print("=" * 60)

# Repartition by the order key into four partitions.
repartitioned_order_sales_df = order_sales_df.repartition(4, "order_id")

print(
    "Repartitioned partition count:",
    repartitioned_order_sales_df.rdd.getNumPartitions()
)

start_time = time.perf_counter()

repartitioned_metrics = repartitioned_order_sales_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("order_id").alias("distinct_orders"),
    F.sum("item_revenue").alias("total_item_revenue"),
    F.sum("freight_cost").alias("total_freight_cost"),
    F.sum("payment_total").alias("total_payment_value"),
).collect()[0]

repartitioned_seconds = time.perf_counter() - start_time

print("\nRepartitioned results:")
print("Rows:", repartitioned_metrics["row_count"])
print("Distinct orders:", repartitioned_metrics["distinct_orders"])
print("Total item revenue:", repartitioned_metrics["total_item_revenue"])
print("Total freight cost:", repartitioned_metrics["total_freight_cost"])
print("Total payment value:", repartitioned_metrics["total_payment_value"])
print(f"Execution time: {repartitioned_seconds:.3f} seconds")

# Validate that repartitioning did not change the results.
assert repartitioned_metrics["row_count"] == baseline_metrics["row_count"]
assert repartitioned_metrics["distinct_orders"] == baseline_metrics["distinct_orders"]

for metric_name in [
    "total_item_revenue",
    "total_freight_cost",
    "total_payment_value",
]:
    baseline_value = baseline_metrics[metric_name]
    repartitioned_value = repartitioned_metrics[metric_name]

    assert abs(baseline_value - repartitioned_value) < 0.01, (
        f"{metric_name} changed after repartitioning: "
        f"{baseline_value} vs {repartitioned_value}"
    )

print("\nRepartitioning correctness checks: PASS")


REPARTITIONED SPARK EXECUTION BENCHMARK
Repartitioned partition count: 4

Repartitioned results:
Rows: 99441
Distinct orders: 99441
Total item revenue: 13591643.699999413
Total freight cost: 2251909.539999944
Total payment value: 16008872.119999759
Execution time: 5.360 seconds

Repartitioning correctness checks: PASS


# Build product-level transformation

In [27]:
from pathlib import Path
import pandas as pd

product_reference_path = PROCESSED_DIR / "product_analytics.csv"
category_reference_path = PROCESSED_DIR / "category_analytics.csv"

for path in [product_reference_path, category_reference_path]:
    print(f"\n{'=' * 60}")
    print(f"File: {path.name}")
    print(f"Exists: {path.exists()}")

    if path.exists():
        reference_df = pd.read_csv(path)
        print("Shape:", reference_df.shape)
        print("Columns:", reference_df.columns.tolist())
        print("\nSample records:")
        display(reference_df.head(3))
    else:
        print("File not found. Check PROCESSED_DIR before proceeding.")




File: product_analytics.csv
Exists: True
Shape: (32729, 7)
Columns: ['product_id', 'product_category_name_english', 'units_sold', 'item_revenue', 'freight_cost', 'distinct_orders', 'distinct_sellers']

Sample records:


,product_id,product_category_name_english,units_sold,item_revenue,freight_cost,distinct_orders,distinct_sellers
0,bb50f2e236e5eea0100680137654686c,health_beauty,195,63885.00,3721.10,187,1
1,6cdd53843498f92890544667809f1595,health_beauty,156,54730.20,4363.79,151,1
2,d6160fb7873f184099d9bc95e30376af,computers,35,48899.34,1426.84,35,1



File: category_analytics.csv
Exists: True
Shape: (72, 7)
Columns: ['product_category_name_english', 'units_sold', 'item_revenue', 'freight_cost', 'distinct_orders', 'distinct_products', 'distinct_sellers']

Sample records:


,product_category_name_english,units_sold,item_revenue,freight_cost,distinct_orders,distinct_products,distinct_sellers
0,health_beauty,9634,1255695.13,181970.65,8800,2430,489
1,watches_gifts,5970,1198185.21,100107.26,5604,1322,98
2,bed_bath_table,11097,1035964.06,204422.07,9399,3022,195


Load products and category translation

In [28]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType
)

# Explicit schemas for the two source datasets.
products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_length", StringType(), True),
    StructField("product_description_length", StringType(), True),
    StructField("product_photos_qty", StringType(), True),
    StructField("product_weight_g", StringType(), True),
    StructField("product_length_cm", StringType(), True),
    StructField("product_height_cm", StringType(), True),
    StructField("product_width_cm", StringType(), True),
])

category_translation_schema = StructType([
    StructField("product_category_name", StringType(), True),
    StructField("product_category_name_english", StringType(), True),
])

products_raw_df = (
    spark.read
    .option("header", True)
    .schema(products_schema)
    .csv(str(RAW_DIR / "olist_products_dataset.csv"))
)

category_translation_raw_df = (
    spark.read
    .option("header", True)
    .schema(category_translation_schema)
    .csv(str(RAW_DIR / "product_category_name_translation.csv"))
)

# Standardize identifiers and category text for reliable joins.
products_clean_df = (
    products_raw_df
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn(
        "product_category_name",
        F.trim(F.col("product_category_name"))
    )
)

category_translation_clean_df = (
    category_translation_raw_df
    .withColumn(
        "product_category_name",
        F.trim(F.col("product_category_name"))
    )
    .withColumn(
        "product_category_name_english",
        F.lower(F.trim(F.col("product_category_name_english")))
    )
)

print("Products rows:", products_clean_df.count())
print("Category translation rows:", category_translation_clean_df.count())

print("\nProduct schema:")
products_clean_df.printSchema()

print("\nCategory translation schema:")
category_translation_clean_df.printSchema()

print("\nProduct sample:")
products_clean_df.show(3, truncate=False)

print("\nCategory translation sample:")
category_translation_clean_df.show(3, truncate=False)

# Validate keys and category mapping uniqueness.
product_key_issues = products_clean_df.filter(
    F.col("product_id").isNull() | (F.col("product_id") == "")
).count()

duplicate_product_ids = (
    products_clean_df
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_category_keys = (
    category_translation_clean_df
    .groupBy("product_category_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\nProduct key issues:", product_key_issues)
print("Duplicate product IDs:", duplicate_product_ids)
print("Duplicate category mapping keys:", duplicate_category_keys)

assert product_key_issues == 0
assert duplicate_product_ids == 0
assert duplicate_category_keys == 0



Products rows: 32951
Category translation rows: 71

Product schema:
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: string (nullable = true)
 |-- product_description_length: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)


Category translation schema:
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)


Product sample:
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_name_length|product_description_length|product_photos_

Prepare order items and join product categories

In [30]:
from pyspark.sql import functions as F

# Keep only orders included by the existing Python product pipeline.
eligible_orders_df = (
    orders_clean_df
    .filter(~F.col("order_status").isin("canceled", "unavailable"))
    .select("order_id", "order_status")
)

# Join items to their order status.
eligible_items_df = (
    order_items_clean_df
    .join(eligible_orders_df, on="order_id", how="inner")
)

# Join product metadata and English category translations.
product_enriched_df = (
    eligible_items_df
    .join(
        products_clean_df.select(
            "product_id", "product_category_name"
        ),
        on="product_id",
        how="inner",
    )
    .join(
        category_translation_clean_df,
        on="product_category_name",
        how="left",
    )
)

# Use the translated category where available.
# Preserve the source category as a fallback, matching the intended
# behavior of retaining products without an English translation.
product_enriched_df = product_enriched_df.withColumn(
    "product_category_name_english",
    F.coalesce(
        F.col("product_category_name_english"),
        F.col("product_category_name"),
    ),
)

print("Eligible orders:", eligible_orders_df.count())
print("Eligible order-item rows:", eligible_items_df.count())
print("Enriched order-item rows:", product_enriched_df.count())

print("\nEnriched schema:")
product_enriched_df.printSchema()

print("\nEnriched sample:")
product_enriched_df.select(
    "order_id",
    "product_id",
    "seller_id",
    "price",
    "freight_value",
    "product_category_name_english",
).show(5, truncate=False)

print("\nJoin validation:")
print(
    "Rows lost during product join:",
    eligible_items_df.count() - product_enriched_df.count(),
)


Eligible orders: 98207
Eligible order-item rows: 112101
Enriched order-item rows: 112101

Enriched schema:
root
 |-- product_category_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- order_status: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)


Enriched sample:
+--------------------------------+--------------------------------+--------------------------------+-----+-------------+-----------------------------+
|order_id                        |product_id                      |seller_id                       |price|freight_value|product_category_name_english|
+--------------------------------+--------------------------------+--------------------------------+----

Build product-level analytics

In [31]:
from pyspark.sql import functions as F

product_analytics_spark_df = (
    product_enriched_df
    .groupBy(
        "product_id",
        "product_category_name_english",
    )
    .agg(
        F.count("order_item_id").alias("units_sold"),
        F.sum("price").alias("item_revenue"),
        F.sum("freight_value").alias("freight_cost"),
        F.countDistinct("order_id").alias("distinct_orders"),
        F.countDistinct("seller_id").alias("distinct_sellers"),
    )
)

print("Product analytics rows:", product_analytics_spark_df.count())

print("\nProduct analytics schema:")
product_analytics_spark_df.printSchema()

print("\nSample product analytics:")
product_analytics_spark_df.orderBy(
    F.desc("item_revenue")
).show(5, truncate=False)

# Basic validation
duplicate_product_groups = (
    product_analytics_spark_df
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_units_sold = product_analytics_spark_df.filter(
    F.col("units_sold").isNull() | (F.col("units_sold") <= 0)
).count()

invalid_revenue = product_analytics_spark_df.filter(
    F.col("item_revenue").isNull() | (F.col("item_revenue") < 0)
).count()

print("\nDuplicate product IDs:", duplicate_product_groups)
print("Invalid units-sold rows:", invalid_units_sold)
print("Invalid revenue rows:", invalid_revenue)

assert duplicate_product_groups == 0
assert invalid_units_sold == 0
assert invalid_revenue == 0



Product analytics rows: 32729

Product analytics schema:
root
 |-- product_id: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- units_sold: long (nullable = false)
 |-- item_revenue: double (nullable = true)
 |-- freight_cost: double (nullable = true)
 |-- distinct_orders: long (nullable = false)
 |-- distinct_sellers: long (nullable = false)


Sample product analytics:
+--------------------------------+-----------------------------+----------+------------------+------------------+---------------+----------------+
|product_id                      |product_category_name_english|units_sold|item_revenue      |freight_cost      |distinct_orders|distinct_sellers|
+--------------------------------+-----------------------------+----------+------------------+------------------+---------------+----------------+
|bb50f2e236e5eea0100680137654686c|health_beauty                |195       |63885.0           |3721.1000000000004|187            |1              

Reconcile Spark product analytics with Python

In [38]:
from pyspark.sql import functions as F

# Identify source categories that have an English translation.
translated_categories_df = (
    category_translation_clean_df
    .select("product_category_name")
    .distinct()
    .withColumn("_has_translation", F.lit(True))
)

# Rebuild enriched items so the source category is available for checking.
product_enriched_df = (
    eligible_items_df
    .join(
        products_clean_df.select(
            "product_id", "product_category_name"
        ),
        on="product_id",
        how="inner",
    )
    .join(
        category_translation_clean_df,
        on="product_category_name",
        how="left",
    )
    .join(
        translated_categories_df,
        on="product_category_name",
        how="left",
    )
    .withColumn(
        "product_category_name_english",
        F.when(
            F.col("_has_translation") == True,
            F.col("product_category_name_english"),
        ).otherwise(F.lit("unknown_or_untranslated")),
    )
    .drop("_has_translation")
)

# Rebuild product-level metrics.
product_analytics_spark_df = (
    product_enriched_df
    .groupBy("product_id", "product_category_name_english")
    .agg(
        F.count("order_item_id").alias("units_sold"),
        F.sum("price").alias("item_revenue"),
        F.sum("freight_value").alias("freight_cost"),
        F.countDistinct("order_id").alias("distinct_orders"),
        F.countDistinct("seller_id").alias("distinct_sellers"),
    )
)

print("Corrected product rows:", product_analytics_spark_df.count())
print(
    "Products using fallback category:",
    product_analytics_spark_df.filter(
        F.col("product_category_name_english")
        == "unknown_or_untranslated"
    ).count(),
)

Corrected product rows: 32729
Products using fallback category: 611


In [39]:
import pandas as pd
from pyspark.sql import functions as F

# Load the existing Python reference output.
product_reference_pd = pd.read_csv(
    PROCESSED_DIR / "product_analytics.csv"
)
product_reference_spark_df = spark.createDataFrame(
    product_reference_pd
)

numeric_metrics = [
    "units_sold",
    "item_revenue",
    "freight_cost",
    "distinct_orders",
    "distinct_sellers",
]

# Give both datasets distinct column names before joining.
spark_product_named_df = product_analytics_spark_df.select(
    "product_id",
    F.col("product_category_name_english").alias("spark_category"),
    *[
        F.col(c).alias(f"spark_{c}")
        for c in numeric_metrics
    ],
)

reference_product_named_df = product_reference_spark_df.select(
    "product_id",
    F.col("product_category_name_english").alias("reference_category"),
    *[
        F.col(c).alias(f"reference_{c}")
        for c in numeric_metrics
    ],
)

comparison_df = spark_product_named_df.join(
    reference_product_named_df,
    on="product_id",
    how="full_outer",
)

# Check missing and extra product IDs.
missing_from_spark = (
    reference_product_named_df
    .select("product_id")
    .join(
        spark_product_named_df.select("product_id"),
        on="product_id",
        how="left_anti",
    )
    .count()
)

extra_in_spark = (
    spark_product_named_df
    .select("product_id")
    .join(
        reference_product_named_df.select("product_id"),
        on="product_id",
        how="left_anti",
    )
    .count()
)

# Check category mismatches.
category_mismatch_condition = ~F.col("spark_category").eqNullSafe(
    F.col("reference_category")
)

category_mismatches = comparison_df.filter(
    category_mismatch_condition
).count()

# Check numeric metrics: tolerance 0.01 for money, exact for counts.
numeric_mismatch_condition = None

for metric in numeric_metrics:
    spark_col = F.col(f"spark_{metric}")
    reference_col = F.col(f"reference_{metric}")

    if metric in ["item_revenue", "freight_cost"]:
        condition = (
            spark_col.isNull()
            | reference_col.isNull()
            | (F.abs(spark_col - reference_col) > F.lit(0.01))
        )
    else:
        condition = (
            spark_col.isNull()
            | reference_col.isNull()
            | (spark_col != reference_col)
        )

    numeric_mismatch_condition = (
        condition
        if numeric_mismatch_condition is None
        else numeric_mismatch_condition | condition
    )

numeric_mismatches = comparison_df.filter(
    numeric_mismatch_condition
).count()

print("Python reference rows:", product_reference_pd.shape[0])
print("Spark product rows:", product_analytics_spark_df.count())
print("Products missing from Spark:", missing_from_spark)
print("Extra products in Spark:", extra_in_spark)
print("Category mismatches:", category_mismatches)
print("Products with numerical mismatches:", numeric_mismatches)

passed = (
    missing_from_spark == 0
    and extra_in_spark == 0
    and category_mismatches == 0
    and numeric_mismatches == 0
)

if passed:
    print("\nPRODUCT RECONCILIATION PASSED")
else:
    print("\nPRODUCT RECONCILIATION FAILED")
    print("\nSample mismatches:")
    comparison_df.filter(
        numeric_mismatch_condition | category_mismatch_condition
    ).show(10, truncate=False)

Python reference rows: 32729
Spark product rows: 32729
Products missing from Spark: 0
Extra products in Spark: 0
Category mismatches: 0
Products with numerical mismatches: 0

PRODUCT RECONCILIATION PASSED


Build Category-level analytics


In [41]:
from pyspark.sql import functions as F

category_analytics_spark_df = (
    product_enriched_df
    .groupBy("product_category_name_english")
    .agg(
        F.count("order_item_id").alias("units_sold"),
        F.sum("price").alias("item_revenue"),
        F.sum("freight_value").alias("freight_cost"),
        F.countDistinct("order_id").alias("distinct_orders"),
        F.countDistinct("product_id").alias("distinct_products"),
        F.countDistinct("seller_id").alias("distinct_sellers"),
    )
)

print("Category analytics rows:", category_analytics_spark_df.count())

print("\nTop categories by revenue:")
category_analytics_spark_df.orderBy(
    F.desc("item_revenue")
).show(10, truncate=False)

print("\nCategory quality checks:")

duplicate_categories = (
    category_analytics_spark_df
    .groupBy("product_category_name_english")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_units = category_analytics_spark_df.filter(
    F.col("units_sold").isNull() | (F.col("units_sold") <= 0)
).count()

print("Duplicate category keys:", duplicate_categories)
print("Invalid units-sold rows:", invalid_units)

assert duplicate_categories == 0
assert invalid_units == 0



Category analytics rows: 72

Top categories by revenue:
+-----------------------------+----------+------------------+------------------+---------------+-----------------+----------------+
|product_category_name_english|units_sold|item_revenue      |freight_cost      |distinct_orders|distinct_products|distinct_sellers|
+-----------------------------+----------+------------------+------------------+---------------+-----------------+----------------+
|health_beauty                |9634      |1255695.1300000174|181970.65000000066|8800           |2430             |489             |
|watches_gifts                |5970      |1198185.2100000128|100107.25999999995|5604           |1322             |98              |
|bed_bath_table               |11097     |1035964.0600000387|204422.07000000082|9399           |3022             |195             |
|sports_leisure               |8590      |979740.9200000191 |167503.71000000002|7673           |2850             |477             |
|computers_accessori

Reconcile category analytics

In [42]:
import pandas as pd
from pyspark.sql import functions as F

# Load the existing Python reference.
category_reference_pd = pd.read_csv(
    PROCESSED_DIR / "category_analytics.csv"
)
category_reference_spark_df = spark.createDataFrame(
    category_reference_pd
)

numeric_metrics = [
    "units_sold",
    "item_revenue",
    "freight_cost",
    "distinct_orders",
    "distinct_products",
    "distinct_sellers",
]

# Rename columns to avoid ambiguous references.
spark_category_named_df = category_analytics_spark_df.select(
    F.col("product_category_name_english").alias("spark_category"),
    *[
        F.col(c).alias(f"spark_{c}")
        for c in numeric_metrics
    ],
)

reference_category_named_df = category_reference_spark_df.select(
    F.col("product_category_name_english").alias("reference_category"),
    *[
        F.col(c).alias(f"reference_{c}")
        for c in numeric_metrics
    ],
)

comparison_df = spark_category_named_df.join(
    reference_category_named_df,
    spark_category_named_df.spark_category
    == reference_category_named_df.reference_category,
    "full_outer",
)

# Count missing and extra category keys.
missing_from_spark = (
    reference_category_named_df
    .join(
        spark_category_named_df,
        reference_category_named_df.reference_category
        == spark_category_named_df.spark_category,
        "left_anti",
    )
    .count()
)

extra_in_spark = (
    spark_category_named_df
    .join(
        reference_category_named_df,
        spark_category_named_df.spark_category
        == reference_category_named_df.reference_category,
        "left_anti",
    )
    .count()
)

# Check counts exactly and monetary metrics within 0.01.
numeric_mismatch_condition = None

for metric in numeric_metrics:
    spark_col = F.col(f"spark_{metric}")
    reference_col = F.col(f"reference_{metric}")

    if metric in ["item_revenue", "freight_cost"]:
        condition = (
            spark_col.isNull()
            | reference_col.isNull()
            | (F.abs(spark_col - reference_col) > F.lit(0.01))
        )
    else:
        condition = (
            spark_col.isNull()
            | reference_col.isNull()
            | (spark_col != reference_col)
        )

    numeric_mismatch_condition = (
        condition if numeric_mismatch_condition is None
        else numeric_mismatch_condition | condition
    )

numeric_mismatches = comparison_df.filter(
    numeric_mismatch_condition
).count()

print("Python reference categories:", category_reference_pd.shape[0])
print("Spark categories:", category_analytics_spark_df.count())
print("Categories missing from Spark:", missing_from_spark)
print("Extra categories in Spark:", extra_in_spark)
print("Categories with numerical mismatches:", numeric_mismatches)

passed = (
    missing_from_spark == 0
    and extra_in_spark == 0
    and numeric_mismatches == 0
)

if passed:
    print("\nCATEGORY RECONCILIATION PASSED")
else:
    print("\nCATEGORY RECONCILIATION FAILED")
    comparison_df.filter(
        numeric_mismatch_condition
    ).show(10, truncate=False)

Python reference categories: 72
Spark categories: 72
Categories missing from Spark: 0
Extra categories in Spark: 0
Categories with numerical mismatches: 0

CATEGORY RECONCILIATION PASSED


# Final testing and edge-case validation

Repo inspection for the existing test structure

In [44]:
from pathlib import Path

print("Repository root:", REPO_ROOT)
print("\nExisting test files:")

test_files = sorted((REPO_ROOT / "tests").glob("*.py"))

for path in test_files:
    print(f"\n--- {path.relative_to(REPO_ROOT)} ---")
    print(path.read_text()[:8000])

print("\nExisting Spark-related source files:")
for path in sorted((REPO_ROOT / "src").rglob("*.py")):
    print(path.relative_to(REPO_ROOT))

print("\nGit status:")
import subprocess

result = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "status", "--short"],
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout or "Working tree clean")

print("\n Inspection only; no files modified.")

Repository root: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline

Existing test files:

--- tests/__init__.py ---
"""Test package."""


--- tests/test_pipeline.py ---
"""Tests for sales analytics pipeline outputs."""

import unittest
from pathlib import Path
import pandas as pd


REPO_ROOT = Path(__file__).resolve().parents[1]
PROCESSED_DIR = REPO_ROOT / "data" / "processed"


class TestPipelineOutputs(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.orders = pd.read_csv(
            PROCESSED_DIR / "order_analytics.csv"
        )
        cls.products = pd.read_csv(
            PROCESSED_DIR / "product_analytics.csv"
        )
        cls.categories = pd.read_csv(
            PROCESSED_DIR / "category_analytics.csv"
        )
        cls.customers = pd.read_csv(
            PROCESSED_DIR / "customer_analytics.csv"
        )
        cls.monthly = pd.read_csv(
            PROCESSED_DIR / "monthly_sales_analytics.csv"
        )


Run final spark assetions


In [45]:
import pandas as pd
from pyspark.sql import functions as F

print("=" * 60)
print("PHASE 3 — SPARK VALIDATION SUMMARY")
print("=" * 60)

# 1. Order-level validation
order_count = order_sales_df.count()
distinct_order_count = order_sales_df.select("order_id").distinct().count()

assert order_count == 99441, f"Unexpected order count: {order_count}"
assert distinct_order_count == order_count, "Duplicate order IDs detected"

print(f"Order-level rows: {order_count}")
print(f"Distinct order IDs: {distinct_order_count}")

# 2. Product-level validation
product_reference = pd.read_csv(PROCESSED_DIR / "product_analytics.csv")
product_count = product_analytics_spark_df.count()
distinct_product_count = (
    product_analytics_spark_df.select("product_id").distinct().count()
)

assert product_count == len(product_reference), (
    f"Product row count mismatch: Spark={product_count}, "
    f"reference={len(product_reference)}"
)
assert distinct_product_count == product_count, "Duplicate product IDs detected"

print(f"\nProduct rows: {product_count}")
print(f"Distinct product IDs: {distinct_product_count}")

# 3. Category-level validation
category_reference = pd.read_csv(PROCESSED_DIR / "category_analytics.csv")
category_count = category_analytics_spark_df.count()
distinct_category_count = (
    category_analytics_spark_df
    .select("product_category_name_english")
    .distinct()
    .count()
)

assert category_count == len(category_reference), (
    f"Category row count mismatch: Spark={category_count}, "
    f"reference={len(category_reference)}"
)
assert distinct_category_count == category_count, "Duplicate categories detected"

print(f"\nCategory rows: {category_count}")
print(f"Distinct category names: {distinct_category_count}")

# 4. Product/category aggregate reconciliation
product_totals = product_analytics_spark_df.agg(
    F.sum("units_sold").alias("units"),
    F.sum("item_revenue").alias("revenue"),
).first()

category_totals = category_analytics_spark_df.agg(
    F.sum("units_sold").alias("units"),
    F.sum("item_revenue").alias("revenue"),
).first()

assert product_totals["units"] == category_totals["units"]
assert abs(product_totals["revenue"] - category_totals["revenue"]) < 0.01

print("\nProduct/category unit totals reconcile.")
print("Product/category revenue totals reconcile.")

print("\nALL PHASE 3 SPARK ASSERTIONS PASSED")
print("No files modified.")

PHASE 3 — SPARK VALIDATION SUMMARY
Order-level rows: 99441
Distinct order IDs: 99441

Product rows: 32729
Distinct product IDs: 32729

Category rows: 72
Distinct category names: 72

Product/category unit totals reconcile.
Product/category revenue totals reconcile.

ALL PHASE 3 SPARK ASSERTIONS PASSED
No files modified.


Create reusable Spark transformation code

In [47]:
from pathlib import Path

transformation_path = SRC_DIR / "sales_pipeline" / "transformation.py"

print("File:", transformation_path)
print("Exists:", transformation_path.exists())

if transformation_path.exists():
    print("\n" + "=" * 70)
    print(transformation_path.read_text(encoding="utf-8"))
    print("=" * 70)
else:
    print("No transformation.py found at the expected path.")

print("\nOther relevant module names:")
for path in sorted((SRC_DIR / "sales_pipeline").glob("*.py")):
    print("-", path.name)

print("\n Inspection only; no files modified.")

File: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/transformation.py
Exists: True

"""Data cleaning and transformation functions."""


Other relevant module names:
- __init__.py
- database.py
- extract.py
- ingestion.py
- pipeline.py
- transform.py
- transform_customers.py
- transform_products.py
- transform_sales.py
- transformation.py
- validate.py
- validation.py

 Inspection only; no files modified.


Crete a dedicated Spark module

In [48]:
from pathlib import Path

files_to_inspect = [
    SRC_DIR / "sales_pipeline" / "transform_products.py",
    SRC_DIR / "sales_pipeline" / "transform.py",
]

for path in files_to_inspect:
    print("\n" + "=" * 80)
    print(f"FILE: {path.relative_to(REPO_ROOT)}")
    print("=" * 80)

    content = path.read_text(encoding="utf-8")
    print(content)

print("\n Inspection only; no files modified.")


FILE: src/sales_pipeline/transform_products.py
"""Product and category analytics transformations."""

from pathlib import Path

import pandas as pd

from .extract import load_csv


EXCLUDED_SALES_STATUSES = {"canceled", "unavailable"}


def build_product_analytics(data_dir):
    """Build product- and category-level sales analytics."""
    data_dir = Path(data_dir)

    items = load_csv(data_dir, "order_items")
    products = load_csv(data_dir, "products")
    translations = load_csv(data_dir, "category_translation")
    orders = load_csv(data_dir, "orders")

    # Join order status to each item before calculating sales metrics.
    sales_items = items.merge(
        orders[["order_id", "order_status"]],
        on="order_id",
        how="left",
        validate="many_to_one",
    )

    # Exclude cancelled and unavailable orders from sales metrics.
    sales_items = sales_items[
        ~sales_items["order_status"].isin(EXCLUDED_SALES_STATUSES)
    ].copy()

    # Add product attribu

Create *spark_transformation.py*

In [49]:
from pathlib import Path

spark_module_path = SRC_DIR / "sales_pipeline" / "spark_transformations.py"

spark_module_code = '''"""Reusable PySpark transformations for sales analytics."""

from pyspark.sql import DataFrame
from pyspark.sql import functions as F


EXCLUDED_SALES_STATUSES = ("canceled", "unavailable")


def build_order_analytics_spark(
    orders: DataFrame,
    order_items: DataFrame,
    payments: DataFrame,
    customers: DataFrame,
) -> DataFrame:
    """Build one row per order with item, payment, customer, and delivery metrics."""

    item_metrics = order_items.groupBy("order_id").agg(
        F.count("order_item_id").alias("item_count"),
        F.countDistinct("product_id").alias("unique_products"),
        F.countDistinct("seller_id").alias("unique_sellers"),
        F.sum("price").alias("item_revenue"),
        F.sum("freight_value").alias("freight_cost"),
    )

    payment_metrics = payments.groupBy("order_id").agg(
        F.sum("payment_value").alias("payment_total"),
        F.count("payment_sequential").alias("payment_records"),
        F.countDistinct("payment_type").alias("payment_types"),
    )

    customer_attributes = customers.select(
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state",
    )

    result = (
        orders
        .join(item_metrics, on="order_id", how="left")
        .join(payment_metrics, on="order_id", how="left")
        .join(customer_attributes, on="customer_id", how="left")
    )

    for column in [
        "item_count",
        "unique_products",
        "unique_sellers",
        "payment_records",
        "payment_types",
    ]:
        result = result.withColumn(
            column,
            F.coalesce(F.col(column), F.lit(0)).cast("long"),
        )

    result = (
        result
        .withColumn(
            "delivery_duration_days",
            (
                F.col("order_delivered_customer_date").cast("long")
                - F.col("order_purchase_timestamp").cast("long")
            ) / F.lit(86400.0),
        )
        .withColumn(
            "estimated_delivery_duration_days",
            (
                F.col("order_estimated_delivery_date").cast("long")
                - F.col("order_purchase_timestamp").cast("long")
            ) / F.lit(86400.0),
        )
        .withColumn(
            "delivery_delay_days",
            (
                F.col("order_delivered_customer_date").cast("long")
                - F.col("order_estimated_delivery_date").cast("long")
            ) / F.lit(86400.0),
        )
        .withColumn(
            "is_delivered",
            F.col("order_status") == F.lit("delivered"),
        )
        .withColumn(
            "is_late_delivery",
            F.when(
                F.col("order_status") == F.lit("delivered"),
                F.coalesce(
                    F.col("delivery_delay_days") > F.lit(0.0),
                    F.lit(False),
                ),
            ).otherwise(F.lit(False)),
        )
    )

    return result


def prepare_sales_items_spark(
    orders: DataFrame,
    order_items: DataFrame,
    products: DataFrame,
    category_translations: DataFrame,
) -> DataFrame:
    """Join eligible order items to products and translated categories."""

    eligible_orders = (
        orders
        .filter(~F.col("order_status").isin(*EXCLUDED_SALES_STATUSES))
        .select("order_id", "order_status")
    )

    translated_categories = (
        category_translations
        .select(
            "product_category_name",
            "product_category_name_english",
        )
    )

    return (
        order_items
        .join(eligible_orders, on="order_id", how="inner")
        .join(
            products.select("product_id", "product_category_name"),
            on="product_id",
            how="left",
        )
        .join(
            translated_categories,
            on="product_category_name",
            how="left",
        )
        .withColumn(
            "product_category_name_english",
            F.coalesce(
                F.col("product_category_name_english"),
                F.lit("unknown_or_untranslated"),
            ),
        )
    )


def build_product_analytics_spark(sales_items: DataFrame) -> DataFrame:
    """Aggregate sales metrics by product and English category."""

    return sales_items.groupBy(
        "product_id",
        "product_category_name_english",
    ).agg(
        F.count("order_item_id").alias("units_sold"),
        F.sum("price").alias("item_revenue"),
        F.sum("freight_value").alias("freight_cost"),
        F.countDistinct("order_id").alias("distinct_orders"),
        F.countDistinct("seller_id").alias("distinct_sellers"),
    )


def build_category_analytics_spark(sales_items: DataFrame) -> DataFrame:
    """Aggregate sales metrics by English category."""

    return sales_items.groupBy(
        "product_category_name_english"
    ).agg(
        F.count("order_item_id").alias("units_sold"),
        F.sum("price").alias("item_revenue"),
        F.sum("freight_value").alias("freight_cost"),
        F.countDistinct("order_id").alias("distinct_orders"),
        F.countDistinct("product_id").alias("distinct_products"),
        F.countDistinct("seller_id").alias("distinct_sellers"),
    )
'''

spark_module_path.write_text(spark_module_code, encoding="utf-8")

print("Created:", spark_module_path.relative_to(REPO_ROOT))
print("File size:", spark_module_path.stat().st_size, "bytes")
print("No existing source files were overwritten.")

Created: src/sales_pipeline/spark_transformations.py
File size: 5191 bytes
No existing source files were overwritten.


Test the reusable product and category functions

In [50]:
import sys
from pyspark.sql import functions as F

# Ensure the repository source directory is importable.
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from sales_pipeline.spark_transformations import (
    prepare_sales_items_spark,
    build_product_analytics_spark,
    build_category_analytics_spark,
)

# Rebuild the enriched sales items using the reusable module.
module_sales_items_df = prepare_sales_items_spark(
    orders_clean_df,
    order_items_clean_df,
    products_clean_df,
    category_translation_clean_df,
)

# Rebuild product and category analytics.
module_product_df = build_product_analytics_spark(module_sales_items_df)
module_category_df = build_category_analytics_spark(module_sales_items_df)

print("Reusable module results:")
print("Enriched item rows:", module_sales_items_df.count())
print("Product rows:", module_product_df.count())
print("Category rows:", module_category_df.count())

# Check output counts.
assert module_sales_items_df.count() == 112101
assert module_product_df.count() == 32729
assert module_category_df.count() == 72

# Check the fallback category behavior.
fallback_count = module_sales_items_df.filter(
    F.col("product_category_name_english") == "unknown_or_untranslated"
).count()

print("Items assigned fallback category:", fallback_count)

# Check category totals against the already validated notebook result.
module_category_totals = module_category_df.agg(
    F.sum("units_sold").alias("units"),
    F.sum("item_revenue").alias("revenue"),
).first()

notebook_category_totals = category_analytics_spark_df.agg(
    F.sum("units_sold").alias("units"),
    F.sum("item_revenue").alias("revenue"),
).first()

assert module_category_totals["units"] == notebook_category_totals["units"]
assert abs(
    module_category_totals["revenue"] - notebook_category_totals["revenue"]
) < 0.01

print("\nReusable product/category functions: PASS")
print("Unit totals and revenue match notebook output.")


Reusable module results:
Enriched item rows: 112101
Product rows: 32729
Category rows: 72
Items assigned fallback category: 1611

Reusable product/category functions: PASS
Unit totals and revenue match notebook output.


Test the resuable order-level function

In [52]:
from pyspark.sql import functions as F
from sales_pipeline.spark_transformations import build_order_analytics_spark

module_order_df = build_order_analytics_spark(
    orders_clean_df,
    order_items_clean_df,
    payments_clean_df,
    customers_clean_df,
)

print("Reusable order-level rows:", module_order_df.count())
print(
    "Distinct order IDs:",
    module_order_df.select("order_id").distinct().count(),
)

assert module_order_df.count() == 99441
assert module_order_df.select("order_id").distinct().count() == 99441

# Compare the shared order-level metrics with the validated notebook output.
metrics_to_compare = [
    "item_count",
    "unique_products",
    "unique_sellers",
    "item_revenue",
    "freight_cost",
    "payment_total",
    "payment_records",
    "payment_types",
    "delivery_duration_days",
    "estimated_delivery_duration_days",
    "delivery_delay_days",
    "is_delivered",
    "is_late_delivery",
]

notebook_order_columns = order_sales_df.select(
    "order_id",
    *[F.col(c).alias(f"notebook_{c}") for c in metrics_to_compare],
)

module_order_columns = module_order_df.select(
    "order_id",
    *[F.col(c).alias(f"module_{c}") for c in metrics_to_compare],
)

order_comparison_df = notebook_order_columns.join(
    module_order_columns,
    on="order_id",
    how="full_outer",
)

missing_or_extra_orders = order_comparison_df.filter(
    F.col("notebook_order_id").isNull()
    | F.col("module_order_id").isNull()
).count() if (
    "notebook_order_id" in order_comparison_df.columns
    and "module_order_id" in order_comparison_df.columns
) else 0

# Count metric mismatches, using a 0.01 tolerance for floating-point values.
mismatch_condition = None

for metric in metrics_to_compare:
    left = F.col(f"notebook_{metric}")
    right = F.col(f"module_{metric}")

    if metric in [
        "item_revenue",
        "freight_cost",
        "payment_total",
        "delivery_duration_days",
        "estimated_delivery_duration_days",
        "delivery_delay_days",
    ]:
        condition = ~left.eqNullSafe(right) & (
            left.isNull()
            | right.isNull()
            | (F.abs(left - right) > F.lit(0.01))
        )
    else:
        condition = ~left.eqNullSafe(right)

    mismatch_condition = (
        condition if mismatch_condition is None
        else mismatch_condition | condition
    )

metric_mismatches = order_comparison_df.filter(
    mismatch_condition
).count()

print("Order-level metric mismatches:", metric_mismatches)

assert metric_mismatches == 0, (
    f"Found {metric_mismatches} order-level metric mismatches."
)

print("\nReusable order-level function: PASS")


Reusable order-level rows: 99441
Distinct order IDs: 99441
Order-level metric mismatches: 0

Reusable order-level function: PASS


Add automated Spark tests

In [53]:
test_path = REPO_ROOT / "tests" / "test_spark_transformations.py"

test_code = '''"""Unit tests for reusable PySpark sales transformations."""

import unittest

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType,
)

from sales_pipeline.spark_transformations import (
    build_order_analytics_spark,
    prepare_sales_items_spark,
    build_product_analytics_spark,
    build_category_analytics_spark,
)


class TestSparkTransformations(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.spark = (
            SparkSession.builder
            .master("local[2]")
            .appName("SalesPipelineSparkTests")
            .config("spark.ui.enabled", "false")
            .getOrCreate()
        )
        cls.spark.sparkContext.setLogLevel("ERROR")

    @classmethod
    def tearDownClass(cls):
        cls.spark.stop()

    def test_product_and_category_aggregations(self):
        orders_schema = StructType([
            StructField("order_id", StringType(), False),
            StructField("order_status", StringType(), False),
        ])
        items_schema = StructType([
            StructField("order_id", StringType(), False),
            StructField("order_item_id", IntegerType(), False),
            StructField("product_id", StringType(), False),
            StructField("seller_id", StringType(), False),
            StructField("price", DoubleType(), False),
            StructField("freight_value", DoubleType(), False),
        ])
        products_schema = StructType([
            StructField("product_id", StringType(), False),
            StructField("product_category_name", StringType(), True),
        ])
        translations_schema = StructType([
            StructField("product_category_name", StringType(), False),
            StructField("product_category_name_english", StringType(), False),
        ])

        orders = self.spark.createDataFrame([
            ("o1", "delivered"),
            ("o2", "delivered"),
            ("o3", "canceled"),
            ("o4", "unavailable"),
        ], orders_schema)

        items = self.spark.createDataFrame([
            ("o1", 1, "p1", "s1", 10.0, 2.0),
            ("o1", 2, "p1", "s1", 10.0, 3.0),
            ("o2", 1, "p2", "s2", 20.0, 4.0),
            ("o3", 1, "p1", "s1", 99.0, 9.0),
            ("o4", 1, "p2", "s2", 88.0, 8.0),
        ], items_schema)

        products = self.spark.createDataFrame([
            ("p1", "cat_a"),
            ("p2", "cat_untranslated"),
        ], products_schema)

        translations = self.spark.createDataFrame([
            ("cat_a", "category_a"),
        ], translations_schema)

        sales_items = prepare_sales_items_spark(
            orders, items, products, translations
        )

        self.assertEqual(sales_items.count(), 3)

        fallback_count = sales_items.filter(
            sales_items.product_category_name_english
            == "unknown_or_untranslated"
        ).count()
        self.assertEqual(fallback_count, 1)

        products_result = build_product_analytics_spark(sales_items)
        self.assertEqual(products_result.count(), 2)

        p1 = products_result.filter(
            products_result.product_id == "p1"
        ).first()
        self.assertEqual(p1["units_sold"], 2)
        self.assertAlmostEqual(p1["item_revenue"], 20.0, places=2)
        self.assertAlmostEqual(p1["freight_cost"], 5.0, places=2)
        self.assertEqual(p1["distinct_orders"], 1)

        categories_result = build_category_analytics_spark(sales_items)
        self.assertEqual(categories_result.count(), 2)

        category_a = categories_result.filter(
            categories_result.product_category_name_english
            == "category_a"
        ).first()
        self.assertEqual(category_a["units_sold"], 2)
        self.assertAlmostEqual(category_a["item_revenue"], 20.0, places=2)

    def test_order_level_metrics_preserve_one_row_per_order(self):
        orders_schema = StructType([
            StructField("order_id", StringType(), False),
            StructField("customer_id", StringType(), False),
            StructField("order_status", StringType(), False),
            StructField("order_purchase_timestamp", TimestampType(), True),
            StructField("order_delivered_customer_date", TimestampType(), True),
            StructField("order_estimated_delivery_date", TimestampType(), True),
        ])
        items_schema = StructType([
            StructField("order_id", StringType(), False),
            StructField("order_item_id", IntegerType(), False),
            StructField("product_id", StringType(), False),
            StructField("seller_id", StringType(), False),
            StructField("price", DoubleType(), False),
            StructField("freight_value", DoubleType(), False),
        ])
        payments_schema = StructType([
            StructField("order_id", StringType(), False),
            StructField("payment_sequential", IntegerType(), False),
            StructField("payment_type", StringType(), False),
            StructField("payment_value", DoubleType(), False),
        ])
        customers_schema = StructType([
            StructField("customer_id", StringType(), False),
            StructField("customer_unique_id", StringType(), False),
            StructField("customer_city", StringType(), False),
            StructField("customer_state", StringType(), False),
        ])

        from datetime import datetime

        orders = self.spark.createDataFrame([
            (
                "o1", "c1", "delivered",
                datetime(2024, 1, 1),
                datetime(2024, 1, 4),
                datetime(2024, 1, 3),
            ),
            (
                "o2", "c2", "canceled",
                datetime(2024, 1, 2),
                None,
                datetime(2024, 1, 5),
            ),
        ], orders_schema)

        items = self.spark.createDataFrame([
            ("o1", 1, "p1", "s1", 10.0, 2.0),
            ("o1", 2, "p2", "s2", 20.0, 3.0),
        ], items_schema)

        payments = self.spark.createDataFrame([
            ("o1", 1, "credit_card", 30.0),
            ("o1", 2, "voucher", 5.0),
        ], payments_schema)

        customers = self.spark.createDataFrame([
            ("c1", "u1", "city_a", "SP"),
            ("c2", "u2", "city_b", "RJ"),
        ], customers_schema)

        result = build_order_analytics_spark(
            orders, items, payments, customers
        )

        self.assertEqual(result.count(), 2)
        self.assertEqual(result.select("order_id").distinct().count(), 2)

        delivered = result.filter(result.order_id == "o1").first()
        self.assertEqual(delivered["item_count"], 2)
        self.assertAlmostEqual(delivered["item_revenue"], 30.0, places=2)
        self.assertAlmostEqual(delivered["payment_total"], 35.0, places=2)
        self.assertEqual(delivered["payment_records"], 2)
        self.assertEqual(delivered["payment_types"], 2)
        self.assertTrue(delivered["is_delivered"])
        self.assertTrue(delivered["is_late_delivery"])

        canceled = result.filter(result.order_id == "o2").first()
        self.assertEqual(canceled["item_count"], 0)
        self.assertEqual(canceled["payment_records"], 0)
        self.assertFalse(canceled["is_delivered"])
        self.assertFalse(canceled["is_late_delivery"])


if __name__ == "__main__":
    unittest.main()
'''

test_path.write_text(test_code, encoding="utf-8")

print("Created:", test_path.relative_to(REPO_ROOT))
print("File size:", test_path.stat().st_size, "bytes")
print("No existing tests were overwritten.")

Created: tests/test_spark_transformations.py
File size: 7441 bytes
No existing tests were overwritten.


Run the Spark unit tests

In [55]:

import os
import subprocess
import sys

test_env = os.environ.copy()
test_env["PYTHONPATH"] = (
    str(SRC_DIR)
    + os.pathsep
    + test_env.get("PYTHONPATH", "")
)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "tests.test_spark_transformations",
        "-v",
    ],
    cwd=str(REPO_ROOT),
    env=test_env,
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

print("Test exit code:", result.returncode)

assert result.returncode == 0, "Spark unit tests failed."


[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/16 17:28:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes wher

Run the complete project test suite

In [57]:

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "psycopg[binary]",
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

assert result.returncode == 0, "PostgreSQL driver installation failed."
print("PostgreSQL driver installation completed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 18.1 MB/s eta 0:00:00

PostgreSQL driver installation completed.


In [77]:

import os
import subprocess
import sys

test_env = os.environ.copy()
test_env["PYTHONPATH"] = (
    str(SRC_DIR)
    + os.pathsep
    + test_env.get("PYTHONPATH", "")
)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        "tests",
        "-v",
    ],
    cwd=str(REPO_ROOT),
    env=test_env,
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

print("Test exit code:", result.returncode)

assert result.returncode == 0, "Full project test suite failed."


[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...

[2/5] Building order analytics...

[3/5] Building product and category analytics...

[4/5] Building customer analytics...

[5/5] Building sales performance analytics...

PIPELINE COMPLETED SUCCESSFULLY
Duration: 0.0 seconds
Validation checks: 1
Order rows: 2
Product rows: 1
Cat

# Document the Spark implementaion

Audit the repo and locate the SQL files

In [67]:

from pathlib import Path
import subprocess

sql_dir = REPO_ROOT / "sql"

expected_sql_files = [
    "01_monthly_revenue_growth.sql",
    "02_order_sales_performance.sql",
    "03_customer_repeat_analysis.sql",
    "04_top_selling_products.sql",
    "05_category_revenue_analysis.sql",
    "06_delivery_performance.sql",
    "07_top_customers_by_revenue.sql",
    "08_seller_coverage_analysis.sql",
]

print("Repository:", REPO_ROOT)
print("SQL directory:", sql_dir)
print("Directory exists:", sql_dir.exists())

print("\n=== EXPECTED SQL FILES ===")
for filename in expected_sql_files:
    path = sql_dir / filename
    print(f"{'FOUND' if path.is_file() else 'MISSING'}  {filename}")

print("\n=== ALL SQL FILES IN THAT DIRECTORY ===")
if sql_dir.exists():
    for path in sorted(sql_dir.glob("*.sql")):
        print(path.name)

print("\n=== TRACKED SQL FILES IN GIT ===")
result = subprocess.run(
    ["git", "ls-tree", "-r", "--name-only", "HEAD", "--", "sql/"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout or "No SQL files tracked in the current HEAD commit.")


Repository: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline
SQL directory: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/sql
Directory exists: True

=== EXPECTED SQL FILES ===
MISSING  01_monthly_revenue_growth.sql
FOUND  02_order_sales_performance.sql
FOUND  03_customer_repeat_analysis.sql
FOUND  04_top_selling_products.sql
FOUND  05_category_revenue_analysis.sql
FOUND  06_delivery_performance.sql
FOUND  07_top_customers_by_revenue.sql
FOUND  08_seller_coverage_analysis.sql

=== ALL SQL FILES IN THAT DIRECTORY ===
02_order_sales_performance.sql
03_customer_repeat_analysis.sql
04_top_selling_products.sql
05_category_revenue_analysis.sql
06_delivery_performance.sql
07_top_customers_by_revenue.sql
08_seller_coverage_analysis.sql

=== TRACKED SQL FILES IN GIT ===
No SQL files tracked in the current HEAD commit.


Re-creating missing file

In [68]:

from pathlib import Path

sql_dir = REPO_ROOT / "sql"
sql_dir.mkdir(parents=True, exist_ok=True)

sql_file = sql_dir / "01_monthly_revenue_growth.sql"

sql_content = """-- Query 01: Monthly Revenue and Growth Analysis
-- Purpose: Measure monthly item revenue and month-over-month growth.
-- Source: public.monthly_sales_analytics
-- Note: This is a recreated query; the original file was not recovered.

WITH monthly_metrics AS (
    SELECT
        order_month,
        order_count,
        item_revenue,
        freight_cost,
        total_items,
        orders_with_revenue,
        average_order_value,

        LAG(item_revenue) OVER (
            ORDER BY order_month
        ) AS previous_month_revenue

    FROM public.monthly_sales_analytics
)
SELECT
    order_month,
    order_count,
    ROUND(item_revenue::numeric, 2) AS item_revenue,
    ROUND(freight_cost::numeric, 2) AS freight_cost,
    total_items,
    orders_with_revenue,
    ROUND(average_order_value::numeric, 2) AS average_order_value,

    ROUND(
        previous_month_revenue::numeric,
        2
    ) AS previous_month_revenue,

    ROUND(
        (item_revenue - previous_month_revenue)::numeric,
        2
    ) AS revenue_change,

    ROUND(
        100.0 * (item_revenue - previous_month_revenue)
        / NULLIF(previous_month_revenue, 0),
        2
    ) AS revenue_growth_percentage

FROM monthly_metrics
ORDER BY order_month;
"""

if sql_file.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing file: {sql_file}"
    )

sql_file.write_text(sql_content, encoding="utf-8")

print("Created:", sql_file)
print("\nFile contents:\n")
print(sql_file.read_text(encoding="utf-8"))


Created: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/sql/01_monthly_revenue_growth.sql

File contents:

-- Query 01: Monthly Revenue and Growth Analysis
-- Purpose: Measure monthly item revenue and month-over-month growth.
-- Source: public.monthly_sales_analytics
-- Note: This is a recreated query; the original file was not recovered.

WITH monthly_metrics AS (
    SELECT
        order_month,
        order_count,
        item_revenue,
        freight_cost,
        total_items,
        orders_with_revenue,
        average_order_value,

        LAG(item_revenue) OVER (
            ORDER BY order_month
        ) AS previous_month_revenue

    FROM public.monthly_sales_analytics
)
SELECT
    order_month,
    order_count,
    ROUND(item_revenue::numeric, 2) AS item_revenue,
    ROUND(freight_cost::numeric, 2) AS freight_cost,
    total_items,
    orders_with_revenue,
    ROUND(average_order_value::numeric, 2) AS average_order_value,

    ROUND(
      

Varify all files

In [69]:

expected_files = [
    "01_monthly_revenue_growth.sql",
    "02_order_sales_performance.sql",
    "03_customer_repeat_analysis.sql",
    "04_top_selling_products.sql",
    "05_category_revenue_analysis.sql",
    "06_delivery_performance.sql",
    "07_top_customers_by_revenue.sql",
    "08_seller_coverage_analysis.sql",
]

missing = [
    name for name in expected_files
    if not (REPO_ROOT / "sql" / name).is_file()
]

for name in expected_files:
    status = "FOUND" if (REPO_ROOT / "sql" / name).is_file() else "MISSING"
    print(f"{status}: {name}")

assert not missing, f"Missing SQL files: {missing}"
print(f"\nPASS: All {len(expected_files)} SQL files are present.")


FOUND: 01_monthly_revenue_growth.sql
FOUND: 02_order_sales_performance.sql
FOUND: 03_customer_repeat_analysis.sql
FOUND: 04_top_selling_products.sql
FOUND: 05_category_revenue_analysis.sql
FOUND: 06_delivery_performance.sql
FOUND: 07_top_customers_by_revenue.sql
FOUND: 08_seller_coverage_analysis.sql

PASS: All 8 SQL files are present.


Document

In [71]:

from pathlib import Path

docs_dir = REPO_ROOT / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

report_path = docs_dir / "phase-3-pyspark-transformation.md"

report = r"""# Phase 3 — PySpark Transformation

## 1. Objective

Implement reusable, tested PySpark transformations for the Sales Analytics Data Engineering Pipeline. The Spark implementation processes the Olist e-commerce dataset and validates its analytics outputs against the existing Python/Pandas pipeline.

## 2. Environment

- Development platform: Google Colab
- Storage and version control: Google Drive and GitHub
- Processing framework: PySpark 4.0.4
- Java runtime: OpenJDK 21
- Spark execution mode: `local[*]`
- Shuffle partitions configured: 4
- Reference outputs: existing processed CSV analytics files
- Database integration from earlier phases: Neon PostgreSQL

## 3. Transformation implementation

Reusable transformations are implemented in `src/sales_pipeline/spark_transformations.py`.

### 3.1 Order analytics

`build_order_analytics_spark()` produces one row per order by independently aggregating order items and payments before joining them to orders and customer attributes.

Metrics include:

- Item count, unique products and unique sellers
- Item revenue and freight cost
- Payment total, payment record count and payment types
- Delivery duration, estimated duration and delivery delay
- Delivery and late-delivery flags

Count metrics are filled with zero when no corresponding item or payment records exist. Financial sums remain null when no source records exist, preserving the reference pipeline's behavior.

### 3.2 Eligible sales items

`prepare_sales_items_spark()` excludes orders with status `canceled` or `unavailable`, joins item records to product attributes and category translations, and assigns `unknown_or_untranslated` when an English category translation is unavailable.

### 3.3 Product analytics

`build_product_analytics_spark()` aggregates units sold, item revenue, freight cost, distinct orders and distinct sellers by product and English category.

### 3.4 Category analytics

`build_category_analytics_spark()` aggregates units sold, item revenue, freight cost, distinct orders, distinct products and distinct sellers by English category.

## 4. Data quality and reconciliation

The notebook performs schema-based reads, standardizes string values, and checks key integrity, duplicates, invalid numeric values and referential integrity.

Observed validation results:

| Dataset/check | Result |
|---|---:|
| Orders | 99,441 rows |
| Order items | 112,650 rows |
| Customers | 99,441 rows |
| Payment records | 103,886 rows |
| Products | 32,951 rows |
| Category translations | 71 rows |
| Eligible sales items | 112,101 rows |
| Order analytics output | 99,441 rows |
| Product analytics output | 32,729 rows |
| Category analytics output | 72 rows |

The observed source-data checks found no invalid order keys, duplicate order IDs, duplicate order-item keys, duplicate customer IDs, duplicate product IDs, duplicate category translation keys, orphan order-item/order links, or orphan order/customer links.

### 4.1 Reference reconciliation

Spark outputs were reconciled against the existing Pandas analytics outputs.

- Order analytics: 99,441 rows; all 13 shared metrics matched.
- Product analytics: 32,729 rows; no missing or extra products, category mismatches, or numerical mismatches.
- Category analytics: 72 categories; no missing or extra categories and no numerical mismatches.
- Product/category units and revenue totals reconciled.

Numeric revenue comparisons used a tolerance of 0.01 where applicable.

## 5. Partitioning experiment

The order analytics aggregation was measured using the baseline DataFrame and after repartitioning by `order_id` to four partitions.

| Configuration | Observed duration |
|---|---:|
| Baseline, 2 input partitions | 2.739 seconds |
| Repartitioned by order ID, 4 partitions | 2.838 seconds |

The aggregate metrics matched between the two runs. The single repartitioned run was approximately 3.6% slower. Based on this small, single-run experiment, no performance benefit was demonstrated, so the baseline configuration was retained.

These timings are exploratory observations, not a statistically reliable benchmark.

## 6. Automated tests

The reusable transformations are covered by `tests/test_spark_transformations.py`, which uses small in-memory Spark DataFrames.

The tests cover product/category aggregation, excluded order statuses, category translation fallback, order-level metric aggregation, one-row-per-order behavior, payment metrics and delivery flags.

The complete project test suite was run successfully:

- Total tests: 14
- Passed: 14
- Failed: 0
- Reported execution time: 54.389 seconds

The test run required installing the `psycopg[binary]` dependency in the test environment.

## 7. SQL analytics

Eight SQL analytics query files are maintained under `sql/`. Seven were restored from the referenced Git commit. `01_monthly_revenue_growth.sql` was recreated because the original could not be located in the available Git history or Google Drive.

The recreated monthly query calculates previous available month revenue, revenue change and month-over-month growth using `LAG()`. Because the monthly table contains gaps, the comparison is against the previous available row, which may not be the immediately preceding calendar month. The recreated query still requires execution against Neon to verify the target table schema and runtime behavior.

## 8. Files delivered in this phase

- `03_PySpark_transformation.ipynb`
- `src/sales_pipeline/spark_transformations.py`
- `tests/test_spark_transformations.py`
- `docs/phase-3-pyspark-transformation.md`
- SQL analytics files under `sql/`

## 9. Conclusion

Phase 3 implements reusable Spark transformations for order-, product- and category-level analytics. The Spark outputs reconciled with the existing Pandas results, and the complete 14-test project suite passed. The partition experiment did not establish a performance gain from increasing the partition count, so the baseline was retained. SQL execution validation and final Git review remain necessary before the phase is considered fully packaged.
"""

if report_path.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing report: {report_path}"
    )

report_path.write_text(report, encoding="utf-8")
print("Created:", report_path)
print("Report size:", report_path.stat().st_size, "bytes")


Created: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/docs/phase-3-pyspark-transformation.md
Report size: 6080 bytes


# Repository integration and Github delivery

Audit dependencies and repository changes

In [72]:

import subprocess
from pathlib import Path

def run_git_audit(args):
    result = subprocess.run(
        ["git", *args],
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout.strip()

print("=== REQUIREMENTS.TXT ===")
requirements_path = REPO_ROOT / "requirements.txt"

if requirements_path.exists():
    print(requirements_path.read_text(encoding="utf-8"))
else:
    print("requirements.txt not found")

print("\n=== PYTHON / SPARK VERSIONS ===")
import sys
import pyspark

print("Python:", sys.version.split()[0])
print("PySpark:", pyspark.__version__)

print("\n=== GIT STATUS ===")
print(run_git_audit(["status", "--short"]))

print("\n=== DIFF SUMMARY ===")
print(run_git_audit(["diff", "--stat"]) or "No tracked-file changes.")

print("\n=== UNTRACKED DELIVERABLES ===")
for relative_path in [
    "03_PySpark_transformation.ipynb",
    "src/sales_pipeline/spark_transformations.py",
    "tests/test_spark_transformations.py",
    "docs/phase-3-pyspark-transformation.md",
]:
    path = REPO_ROOT / relative_path
    print(
        f"{'FOUND' if path.is_file() else 'MISSING'}: {relative_path}"
    )


=== REQUIREMENTS.TXT ===
pandas
numpy
pytest
psycopg[binary]


=== PYTHON / SPARK VERSIONS ===
Python: 3.13.16
PySpark: 4.0.4

=== GIT STATUS ===
M 02_postgresql_data_warehouse.ipynb
?? 03_PySpark_transformation.ipynb
?? docs/
?? sql/
?? src/sales_pipeline/spark_transformations.py
?? tests/test_spark_transformations.py

=== DIFF SUMMARY ===
02_postgresql_data_warehouse.ipynb | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)

=== UNTRACKED DELIVERABLES ===
FOUND: 03_PySpark_transformation.ipynb
FOUND: src/sales_pipeline/spark_transformations.py
FOUND: tests/test_spark_transformations.py
FOUND: docs/phase-3-pyspark-transformation.md


Update requirement.txt

In [73]:

from pathlib import Path

requirements_path = REPO_ROOT / "requirements.txt"

current = requirements_path.read_text(encoding="utf-8")
required_dependency = "pyspark>=4.0,<4.1"

if required_dependency in current:
    print("PySpark dependency is already documented.")
else:
    updated = current.rstrip() + "\n" + required_dependency + "\n"
    requirements_path.write_text(updated, encoding="utf-8")
    print("Added PySpark dependency.")

print("\nUpdated requirements.txt:\n")
print(requirements_path.read_text(encoding="utf-8"))


Added PySpark dependency.

Updated requirements.txt:

pandas
numpy
pytest
psycopg[binary]
pyspark>=4.0,<4.1



Validate the recreated SQL query against Neon

In [74]:

from pathlib import Path
import os
import psycopg

# Read the secret without displaying it.
try:
    from google.colab import userdata
    database_url = userdata.get("NEON_DB_URL")
except Exception:
    database_url = os.environ.get("NEON_DB_URL")

if not database_url:
    raise RuntimeError("NEON_DB_URL is unavailable in this runtime.")

sql_path = REPO_ROOT / "sql" / "01_monthly_revenue_growth.sql"
query = sql_path.read_text(encoding="utf-8")

with psycopg.connect(database_url, connect_timeout=15) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND table_name = 'monthly_sales_analytics'
            ORDER BY ordinal_position
        """)
        columns = [row[0] for row in cur.fetchall()]

        print("Monthly analytics table columns:", columns)

        if not columns:
            raise RuntimeError(
                "public.monthly_sales_analytics was not found."
            )

        cur.execute(query)
        rows = cur.fetchall()
        column_names = [desc[0] for desc in cur.description]

print("\nQuery executed successfully.")
print("Output columns:", column_names)
print("Rows returned:", len(rows))
print("Sample rows:")
for row in rows[:5]:
    print(row)



Monthly analytics table columns: ['order_month', 'order_count', 'item_revenue', 'freight_cost', 'total_items', 'orders_with_revenue', 'average_order_value']

Query executed successfully.
Output columns: ['order_month', 'order_count', 'item_revenue', 'freight_cost', 'total_items', 'orders_with_revenue', 'average_order_value', 'previous_month_revenue', 'revenue_change', 'revenue_growth_percentage']
Rows returned: 24
Sample rows:
('2016-09', 2, Decimal('207.86'), Decimal('71.83'), 5.0, 2, Decimal('103.93'), None, None, None)
('2016-10', 293, Decimal('44507.30'), Decimal('6847.22'), 342.0, 290, Decimal('153.47'), Decimal('207.86'), Decimal('44299.44'), Decimal('21312.15'))
('2016-12', 1, Decimal('10.90'), Decimal('8.72'), 1.0, 1, Decimal('10.90'), Decimal('44507.30'), Decimal('-44496.40'), Decimal('-99.98'))
('2017-01', 787, Decimal('120098.27'), Decimal('16845.19'), 953.0, 787, Decimal('152.60'), Decimal('10.90'), Decimal('120087.37'), Decimal('1101718.99'))
('2017-02', 1718, Decimal('244

Fianl git review

In [75]:

import subprocess
from pathlib import Path

def git_check(args):
    result = subprocess.run(
        ["git", *args],
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout.strip()

print("=== FINAL FILE CHECK ===")

required_files = [
    "requirements.txt",
    "03_PySpark_transformation.ipynb",
    "src/sales_pipeline/spark_transformations.py",
    "tests/test_spark_transformations.py",
    "docs/phase-3-pyspark-transformation.md",
]

for name in required_files:
    path = REPO_ROOT / name
    print(f"{'FOUND' if path.is_file() else 'MISSING'}: {name}")

sql_files = sorted((REPO_ROOT / "sql").glob("*.sql"))
print(f"\nSQL files: {len(sql_files)}")
for path in sql_files:
    print(" ", path.name)

print("\n=== GIT STATUS ===")
print(git_check(["status", "--short"]))

print("\n=== NOTEBOOK CHANGE SUMMARY ===")
print(git_check(["diff", "--stat", "--", "02_postgresql_data_warehouse.ipynb"]))

print("\n=== REQUIREMENTS DIFF ===")
print(git_check(["diff", "--", "requirements.txt"]) or "No tracked requirements changes.")


=== FINAL FILE CHECK ===
FOUND: requirements.txt
FOUND: 03_PySpark_transformation.ipynb
FOUND: src/sales_pipeline/spark_transformations.py
FOUND: tests/test_spark_transformations.py
FOUND: docs/phase-3-pyspark-transformation.md

SQL files: 8
  01_monthly_revenue_growth.sql
  02_order_sales_performance.sql
  03_customer_repeat_analysis.sql
  04_top_selling_products.sql
  05_category_revenue_analysis.sql
  06_delivery_performance.sql
  07_top_customers_by_revenue.sql
  08_seller_coverage_analysis.sql

=== GIT STATUS ===
M 02_postgresql_data_warehouse.ipynb
 M requirements.txt
?? 03_PySpark_transformation.ipynb
?? docs/
?? sql/
?? src/sales_pipeline/spark_transformations.py
?? tests/test_spark_transformations.py

=== NOTEBOOK CHANGE SUMMARY ===
02_postgresql_data_warehouse.ipynb | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)

=== REQUIREMENTS DIFF ===
diff --git a/requirements.txt b/requirements.txt
index e4fb19f..dea6f4e 100644
--- a/requirements.txt
+++ b/requirements.txt
@@ -2,3

Inspect the PostgreSQL notebook change

In [76]:

import subprocess

result = subprocess.run(
    [
        "git", "diff", "--",
        "02_postgresql_data_warehouse.ipynb",
    ],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)

print(result.stdout or "No notebook diff.")
if result.stderr:
    print(result.stderr)

assert result.returncode == 0, "Could not inspect notebook diff."


diff --git a/02_postgresql_data_warehouse.ipynb b/02_postgresql_data_warehouse.ipynb
index ee56bcc..7c758d6 100644
--- a/02_postgresql_data_warehouse.ipynb
+++ b/02_postgresql_data_warehouse.ipynb
@@ -1 +1 @@
-{"nbformat":4,"nbformat_minor":0,"metadata":{"colab":{"provenance":[],"authorship_tag":"ABX9TyOM8HvkweKVT5tezhnPF2iH"},"kernelspec":{"name":"python3","display_name":"Python 3"},"language_info":{"name":"python"}},"cells":[{"cell_type":"code","execution_count":2,"metadata":{"colab":{"base_uri":"https://localhost:8080/"},"id":"nzu8PchbUAQD","executionInfo":{"status":"ok","timestamp":1792086552188,"user_tz":-330,"elapsed":7098,"user":{"displayName":"Vipul Kumawat","userId":"10244865491502480161"}},"outputId":"9dc5b818-736d-4665-96d6-15a4cd7f8dd4"},"outputs":[{"output_type":"stream","name":"stdout","text":["Collecting psycopg[binary]\n","  Downloading psycopg-3.3.6-py3-none-any.whl.metadata (4.4 kB)\n","Collecting psycopg-binary==3.3.6 (from psycopg[binary])\n","  Downloading psycopg_